# BetaTour run v25 (pilot): LKH'nin attığı deneme turlarını birleştirmek

**Fikir.** LKH-3 her denemede bir yerel en iyi tur üretir, bunu en iyi turuyla birleştirir ve yalnızca en iyi turu tutar; deneme turunun kendisi atılır. 20 Kasım 2026 ön denemesinde (tek örnek, 10.000 düğüm, geliştirme ortamı) altı sürecin son 17'şer deneme turunu GA-EAX ile birleştirmek, altı sürecin en iyisinden %0,023 daha kısa tur verdi; yalnızca altı en iyi turu birleştirmek %0,013 verdi. 50.000 düğümlü tek örnekte fark %0,06 dolayındaydı, ama o karşılaştırma eşit sürede yapılmamıştı. Bu pilot aynı şeyi yeni örneklerde ve **eşit duvar saati süresinde** ölçer.

**Bu bir pilottur (keşif amaçlı).** Sonuç iyi de çıksa makaleye girecek bir iddia için, kuralı önceden yazılmış ve çalıştırılmadan önce depoya gönderilmiş ayrı bir doğrulama koşusu gerekir.

**Kollar.** Her kolun toplam süresi örneğin bütçesidir ve hepsi aynı makinede koşar. Bütün zamanlar duvar saatidir.

| Kol | Ne yapar |
|---|---|
| `lkh6` | Altı LKH-3.0.14 süreci (POPMUSIC-5, tohum 101–106) bütçenin sonuna kadar sürer; altısının en iyisi. Önceki koşulardaki `pop_full` ayarları. |
| `trials_50` (birincil) | Aynı altı süreç bütçenin yarısında durur. Her sürecin en iyi turu ve son 16 atılan deneme turu (102 tur) GA-EAX'in popülasyonu olur; GA-EAX kalan sürede tek çekirdekte koşar. |
| `best6_50` (kontrol) | Aynısı, ama popülasyon yalnızca altı en iyi turdur. Atılan turların katkısını ayırır. |
| `trials_33`, `trials_67`, `best6_33`, `best6_67` | Aynı iki kol, LKH bütçenin üçte birinde ya da üçte ikisinde durdurulursa. |
| `cascade` | Bilinen melez: altı işçi LKH'nin tek denemelik 100 turunu üretir, GA-EAX bu turlardan başlar ve bütçe sonunda durur. |
| `eax_scratch` (referans) | Sıfırdan GA-EAX, yazarın varsayılan ayarları (popülasyon 300), tek süreç, 6 × bütçe saniye (altı LKH sürecinin toplam CPU süresi). Yalnızca 20.000 düğüme kadar. |

`lkh6` ile birleştirme kolları aynı kayıtlı koşudan çıkar: LKH'nin kayıt eden kopyası her denemenin turunu ve her yeni en iyi turu duvar saati damgasıyla dosyaya yazar; arama değişmez (öz sınama bunu bu makinede denetler). Birleştirme kollarında GA-EAX'e verilen süre = bütçe − LKH'nin durdurulduğu an − popülasyonu hazırlama süresi. GA-EAX tek çekirdekte koşar; LKH durduktan sonra diğer beş çekirdek boş kalır.

**Örnekler (21).** Daha önce kullanılmamış 12 örnek 10.000 düğüm (6 uniform, 6 kümeli; bütçe 120 s), optimumu bilinen 5 büyük TSPLIB örneği (rl11849, usa13509, brd14051, d15112, d18512; bütçe 0,02 × düğüm sayısı), 4 örnek 50.000 düğüm (2 uniform, 2 kümeli; bütçe 600 s).

**Önceden yazılan karar kuralı** (`experiments/PROTOCOL_v25.md`):

- **D1 (tekrar):** 10.000 düğümlü 12 örnekte `trials_50`, `lkh6`'dan daha çok örnekte kısa ve kesin iki yönlü işaret testi p < 0,05 (eşitlik yoksa en az 10 örnek).
- **D2 (boyutla büyüme):** 50.000 düğümlü dört örneğin hepsinde `trials_50` daha kısa ve ortalama fark 10.000 düğümdekinden büyük.
- D1 ve D2 birlikte sağlanırsa doğrulama koşusu hazırlanır. Sağlanmazsa sonuç depoya kısa bir kayıt olarak yazılır.
- D3 (bilgi amaçlı): 10.000 düğümde `trials_50`, `best6_50`'den daha çok örnekte kısa ve p < 0,05.

**Çalıştırma.** Colab'da **L4 GPU** ve **Yüksek RAM** seçin (12 vCPU, 53 GB; GPU kullanılmaz, bu ortam yalnızca çekirdek sayısı ve bellek için gerekir; A100 de olur ama gereksiz yere pahalıdır). Sonra *Çalışma zamanı → Tümünü çalıştır*. Drive izni istendiğinde onaylayın ve sekmeyi açık tutun. Bağlantı koparsa aynı ortam türüyle yeniden *Tümünü çalıştır* deyin: biten örnekler Drive'daki `MyDrive/BetaTour/v25_trialtours21` klasöründen okunur ve atlanır; yarım kalan örnek baştan koşar.

**Tahmini süre.**

| Aşama | Süre |
|---|---|
| Kurulum: depo, LKH'nin iki derlemesi, GA-EAX'in iki derlemesi, TSPLIB dosyaları | ≈ 4–6 dk |
| Öz sınama | ≈ 1 dk |
| 10.000 düğüm, örnek başına ≈ 6 dk (kayıtlı LKH 2 dk, birleştirmeler 1,3 dk, bilinen melez 2 dk) × 12 | ≈ 1,2 saat |
| TSPLIB, örnek başına ≈ 11–17 dk × 5 | ≈ 1,1 saat |
| 50.000 düğüm, örnek başına ≈ 28 dk (kayıtlı LKH 10 dk, birleştirmeler 6,7 dk, bilinen melez 10 dk) × 4 | ≈ 1,9 saat |
| Sıfırdan GA-EAX referansı (altışar örnek paralel; 12 + 12 + 37 dk) | ≈ 1,0 saat |
| Çözümleme ve ZIP | ≈ 1 dk |
| **Toplam** | **≈ 5,3 saat** |

İlk karar (D1) yaklaşık 1,3 saat sonra, ilk 12 örnek bitince ara raporda görülür. Daha kısa oturumlar için ilk hücrede `MAX_NEW_INSTANCES = 12` yazın (≈ 1,3 saat); defteri yeniden çalıştırmak kalan örneklerle sürer. Referans koşuları ana örneklerin hepsi bitince başlar; istemezseniz `RUN_REFERENCE = False` yazın (≈ 1 saat kısalır). Sonunda `v25_trialtours21_results.zip` tarayıcıya iner (Drive'da da durur); bu dosyayı gönderin.

In [ ]:
# Colab çalışma ortamı: Yüksek RAM ve 12 vCPU (L4 ya da A100 GPU + Yüksek RAM). GPU kullanılmaz; bu ortam türü
# yalnızca çekirdek sayısı ve bellek için seçilir. Deney bitene kadar aynı ortam türünü kullanın.
EXPERIMENT_NAME = "v25_trialtours21"
MAX_NEW_INSTANCES = None      # örn. 12: deneyi kısa oturumlara böler; biten örnekler Drive'dan okunur ve atlanır
SAVE_TO_DRIVE = True
ALLOW_HARDWARE_CHANGE = False  # True: sonraki oturum başka makine türünde sürebilir (rapora not düşülür)
RUN_REFERENCE = True          # sıfırdan GA-EAX referans koşuları (eşit toplam CPU süresi); ana örnekler bitince koşar

W = 6                         # paralel LKH süreci (tohum) sayısı
SEEDS = list(range(101, 107))
# (grup, düğüm sayısı, örnek sayısı, aile, üretici tohumu, bütçe s). Tohumların hiçbiri önceki koşularda ya da ön
# denemede kullanılmadı. Bütçe: düğüm başına 12 ms (10.000 düğümde 120 s, önceki koşulardaki gibi).
SYNTHETIC = [("uni10k", 10000, 6, "uniform", 9111, 120.0), ("clu10k", 10000, 6, "clustered", 9212, 120.0),
             ("uni50k", 50000, 2, "uniform", 9515, 600.0), ("clu50k", 50000, 2, "clustered", 9616, 600.0)]
# Optimumu bilinen büyük TSPLIB örnekleri (EUC_2D); bütçe run v22'deki gibi 0,02 x düğüm sayısı saniye.
TSPLIB_NAMES = ["rl11849", "usa13509", "brd14051", "d15112", "d18512"]
K_LIST = 5
SPLITS = [1 / 3, 1 / 2, 2 / 3]  # LKH turlarının alındığı an (bütçenin kesri); kalan süre GA-EAX'e verilir
PRIMARY_SPLIT = 1 / 2         # önceden belirlenen birincil bölme
PER_PROCESS = 17              # süreç başına popülasyona giren tur: en iyi tur + son 16 atılan deneme turu
EAX_KIDS = 30                 # GA-EAX: ebeveyn çifti başına çocuk sayısı (yazarın önerdiği değer)
EAX_SCRATCH_POP = 300         # sıfırdan GA-EAX referansının popülasyonu (yazarın önerdiği değer)
EAX_NORMAL_MAX_N = 20000      # bu boyuta kadar GA-EAX'in "Normal" sürümü, üstünde "Large" sürümü (bellek)
REFERENCE_MAX_N = 20000       # sıfırdan GA-EAX referansı yalnızca bu boyuta kadar koşar
CASCADE_TOURS = 100           # bilinen melez: LKH'nin tek denemelik tur sayısı
CASCADE_GEN_FRACTION = 0.4    # bilinen melezde tur üretimine ayrılan en uzun süre (bütçenin kesri)
CASCADE_SEED0 = 2001
STOP_PAD_S = 2.0              # kayıt eden LKH süreçleri bütçeden bu kadar sonra dışarıdan durdurulur
REQUIRE_12_VCPUS = True
MIN_RAM_GB = 24.0
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CODE_SHA256_EXPECTED = "198c8044b2ff3975234ededd3e222ea37213d2f109e0ee03cc078fe8a2bf0eef"   # sonraki hücrenin yazdığı byprod_v25.py dosyasının özeti
NOTEBOOK_BUILD = "8e6b71c08d3c8c7c273569518f7177d5d4eedc84bd97fe32a8ebb1b845c6c13c"      # bu defterin bütün kod hücrelerinin özeti

import os

TEST_MODE = os.environ.get("BETATOUR_V25_TEST") == "1"    # yalnızca yerel öz sınama; Colab'da kapalıdır
if TEST_MODE:
    EXPERIMENT_NAME = "v25_selftest"
    SAVE_TO_DRIVE = False
    REQUIRE_12_VCPUS = False
    MIN_RAM_GB = 0.0
    W = int(os.environ.get("BETATOUR_V25_W", "2"))
    SEEDS = SEEDS[:W]
    SYNTHETIC = [("uni10k", int(os.environ.get("BETATOUR_V25_N", "1200")), 1, "uniform", 9111,
                  float(os.environ.get("BETATOUR_V25_BUDGET", "12"))),
                 ("clu50k", 2500, 1, "clustered", 9616, 15.0)]
    TSPLIB_NAMES = ["test_euc900"]                        # yerel olarak üretilen TSPLIB biçimli dosya
    PER_PROCESS = int(os.environ.get("BETATOUR_V25_PER_PROCESS", "5"))
    EAX_SCRATCH_POP = 30
    EAX_NORMAL_MAX_N = 2000                               # 2.500 düğümlü örnek "Large" sürümünü sınar
    REFERENCE_MAX_N = 2000
    CASCADE_TOURS = int(os.environ.get("BETATOUR_V25_CASCADE_TOURS", "12"))
assert W == len(SEEDS) and PRIMARY_SPLIT in SPLITS and PER_PROCESS >= 2

In [ ]:
%%writefile byprod_v25.py
"""Helpers of run v25 (pilot): the trial tours that LKH-3 discards, recorded and recombined with GA-EAX.

Every trial of LKH-3 ends in a locally optimal tour. LKH-3 merges it with its best tour and keeps only the best
tour. This module

  * patches FindTour.c of LKH-3.0.14 so that a copy of LKH-3 writes the tour of every trial and every new best tour
    to a file (nothing of the search state is changed),
  * patches env.cpp of GA-EAX (Nagata and Kobayashi, 2013) so that it prints a wall-clock time stamp per generation
    and stops at a wall-clock deadline with its best tour,
  * patches evaluator.cpp of GA-EAX so that the lists of nearest cities are found with a grid instead of a scan of
    all cities for every list entry (the same lists; at 50,000 nodes the scan takes about a minute),
  * reads the recorded tours, builds populations from them and runs GA-EAX on a population,
  * holds the paired statistics used by the analysis cell.

The patches are applied by exact string replacement after the SHA-256 of the original file has been compared with
the value below, so another source version is refused and never patched wrongly.

Pure Python and NumPy; tsplab is not imported here.
"""

from __future__ import annotations

import hashlib
import os
import re
import subprocess
import time
from math import comb

import numpy as np

VERSION = "v25-byprod-1"

# ----------------------------------------------------------------------------------------------- source patches
LKH_FINDTOUR_SHA256 = "567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee"      # LKH-3.0.14
EAX_COMMIT = "015dfbe9f267230f78787bd244af393ffc018900"                                     # GA-EAX, 2021-12-29
EAX_ENV_SHA256 = {"Normal": "f923055933b26b3bde02bd1467abca92e99a357d590e62f8044f047ffc8fa6e5",
                  "Large": "5c58951f29477c34f0bd10364147e70ce475899799324775c7df98ddabeae35c"}
EAX_EVALUATOR_SHA256 = {"Normal": "85c07c041c7ea32d42d3b180d1eed30d25707bfeb01393c010803ffbcdcd540c",
                        "Large": "dd1582b36c0c595209cf6d67c287f20000dc24737cd252b474bb03e1763b9767"}
EAX_SOURCES = ["main.cpp", "env.cpp", "cross.cpp", "evaluator.cpp", "indi.cpp", "rand.cpp", "kopt.cpp", "sort.cpp"]
# SHA-256 over the names and contents of all .cpp and .h files of a version (see source_digest).
EAX_TREE_SHA256 = {"Normal": "d4d016b831745a54bd39c458c447e55ac4039e8606220d6fbb2b4ecd130b5813",
                   "Large": "bc5785bdb728b8d46472480eaa5e962faa547b99b02b4d47055ac58971af92af"}

_LKH_FUNCTION = r'''
#include <sys/time.h>

/* Run v25 (pilot): record tours without changing the search.
 * Kind 0: the locally optimal tour of the trial that has just ended (before it is merged with the best tour).
 * Kind 1: the new best tour of the run.
 * Record: five 64-bit integers (kind, trial, cost, milliseconds of LKH's clock since the start of the search,
 * milliseconds of the wall clock since the epoch), then Dimension 32-bit node numbers in tour order. */
static void BetaTourRecord(int Kind, GainType RecordCost, double RecordEntryTime)
{
    static FILE *File = 0;
    static int Tried = 0;
    static int *Buffer = 0;
    long long Head[5];
    struct timeval Now;
    int i;

    if (!Tried) {
        char *Name = getenv("BETATOUR_TRIAL_DUMP");
        Tried = 1;
        if (Name)
            File = fopen(Name, "wb");
        if (File)
            Buffer = (int *) malloc((size_t) Dimension * sizeof(int));
    }
    if (!File || !Buffer)
        return;
    if (Kind == 0) {
        Node *N = FirstNode;
        for (i = 0; i < Dimension; i++) {
            Buffer[i] = N->Id;
            N = N->Suc;
        }
    } else
        for (i = 0; i < Dimension; i++)
            Buffer[i] = BetterTour[i + 1];
    gettimeofday(&Now, 0);
    Head[0] = Kind;
    Head[1] = Trial;
    Head[2] = (long long) RecordCost;
    Head[3] = (long long) (1000.0 * (GetTime() - RecordEntryTime));
    Head[4] = (long long) Now.tv_sec * 1000 + Now.tv_usec / 1000;
    fwrite(Head, sizeof(long long), 5, File);
    fwrite(Buffer, sizeof(int), (size_t) Dimension, File);
    fflush(File);
}
'''

_LKH_EDITS = [
    ('#include "LKH.h"\n',
     '#include "LKH.h"\n' + _LKH_FUNCTION),
    ("        Cost = LinKernighan();\n        if (GetTime() - EntryTime < TimeLimit &&",
     "        Cost = LinKernighan();\n        BetaTourRecord(0, Cost, EntryTime);\n"
     "        if (GetTime() - EntryTime < TimeLimit &&"),
    ("            RecordBetterTour();\n            if ((BetterPenalty < BestPenalty ||",
     "            RecordBetterTour();\n            BetaTourRecord(1, BetterCost, EntryTime);\n"
     "            if ((BetterPenalty < BestPenalty ||"),
]

_EAX_OLD = '    printf( "%d: %d %lf\\n", fCurNumOfGen, fBestValue, fAverageValue );\n'
_EAX_NEW = r'''    {
      /* Run v25 (pilot): wall-clock time stamp of every generation and an optional wall-clock deadline. */
      struct timeval pilotNow;
      gettimeofday( &pilotNow, NULL );
      double pilotTime = (double)pilotNow.tv_sec + 1e-6 * (double)pilotNow.tv_usec;
      printf( "%d: %d %lf %.3f %d\n", fCurNumOfGen, fBestValue, fAverageValue, pilotTime, fStage );
      fflush( stdout );
      const char* pilotDeadline = getenv( "EAX_DEADLINE" );
      if( pilotDeadline != NULL && pilotTime >= atof( pilotDeadline ) ) break;
    }
'''
_EAX_HEAD = "#include <stdio.h>\n#include <stdlib.h>\n#include <sys/time.h>\n"

_EAX_NEAREST = r'''
#include <math.h>
#include <stdlib.h>
#include <vector>
#include <algorithm>
#include <utility>

/* Run v25 (pilot): the nearest cities of every city, found with a grid instead of a scan of all cities.
 * The result is the list of the original loops: ascending rounded distance and, among equal distances, the larger
 * city number first. nearDis may be NULL. Requires N > K + 1. */
static void PilotNearest( const double* x, const double* y, int N, int K, int** nearCity, int** nearDis )
{
  double xmin = x[0], xmax = x[0], ymin = y[0], ymax = y[0];
  for( int i = 1; i < N; ++i ){
    if( x[i] < xmin ) xmin = x[i];
    if( x[i] > xmax ) xmax = x[i];
    if( y[i] < ymin ) ymin = y[i];
    if( y[i] > ymax ) ymax = y[i];
  }
  double span = ( xmax - xmin > ymax - ymin ) ? xmax - xmin : ymax - ymin;
  int G = (int)sqrt( (double)N / 2.0 );
  if( G < 1 ) G = 1;
  double cell = span / (double)G;
  if( !( cell > 0.0 ) ){ cell = 1.0; G = 1; }
  std::vector<int> cx( N ), cy( N ), head( G * G + 1, 0 ), order( N );
  for( int i = 0; i < N; ++i ){
    int a = (int)( ( x[i] - xmin ) / cell ), b = (int)( ( y[i] - ymin ) / cell );
    cx[i] = a < 0 ? 0 : ( a >= G ? G - 1 : a );
    cy[i] = b < 0 ? 0 : ( b >= G ? G - 1 : b );
    ++head[ cx[i] * G + cy[i] + 1 ];
  }
  for( int c = 0; c < G * G; ++c ) head[ c + 1 ] += head[ c ];
  {
    std::vector<int> next( head.begin(), head.end() - 1 );
    for( int i = 0; i < N; ++i ) order[ next[ cx[i] * G + cy[i] ]++ ] = i;
  }
  std::vector< std::pair<int,int> > cand;
  for( int i = 0; i < N; ++i ){
    cand.clear();
    for( int r = 0; ; ++r ){
      for( int a = cx[i] - r; a <= cx[i] + r; ++a ){
        if( a < 0 || a >= G ) continue;
        for( int b = cy[i] - r; b <= cy[i] + r; ++b ){
          if( b < 0 || b >= G ) continue;
          if( abs( a - cx[i] ) != r && abs( b - cy[i] ) != r ) continue;   /* the cells of ring r only */
          for( int p = head[ a * G + b ]; p < head[ a * G + b + 1 ]; ++p ){
            int j = order[ p ];
            if( j == i ) continue;
            int d = (int)(sqrt((x[i]-x[j])*(x[i]-x[j])+(y[i]-y[j])*(y[i]-y[j]))+0.5);
            cand.push_back( std::make_pair( d, -j ) );
          }
        }
      }
      if( cx[i] - r <= 0 && cx[i] + r >= G - 1 && cy[i] - r <= 0 && cy[i] + r >= G - 1 ) break;   /* whole grid */
      if( (int)cand.size() >= K ){
        std::nth_element( cand.begin(), cand.begin() + ( K - 1 ), cand.end() );
        /* A city outside the rings 0..r is farther than r * cell from city i. */
        if( (double)r * cell > (double)cand[ K - 1 ].first + 1.0 ) break;
      }
    }
    std::partial_sort( cand.begin(), cand.begin() + K, cand.end() );
    nearCity[ i ][ 0 ] = i;
    if( nearDis != NULL ) nearDis[ i ][ 0 ] = 0;
    for( int k = 1; k <= K; ++k ){
      nearCity[ i ][ k ] = -cand[ k - 1 ].second;
      if( nearDis != NULL ) nearDis[ i ][ k ] = cand[ k - 1 ].first;
    }
  }
}
'''
_EAX_EVALUATOR_HEAD = '#ifndef __EVALUATOR__\n#include "evaluator.h"\n#endif\n'
_EAX_NORMAL_START = "  for( ci = 0; ci < Ncity; ++ci )\n  {\n    for( j3 = 0; j3 < Ncity; ++j3 ) checkedN[ j3 ] = 0;"
_EAX_LARGE_START = '  for( int i = 0; i < Ncity ; ++i )\n  {\n    if( strcmp( fType, "EUC_2D" ) == 0  ) {'
_EAX_LARGE_END = "  delete [] DisTmp;\n}"


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _replace_once(text, old, new):
    assert text.count(old) == 1, "the source does not hold the expected line exactly once: " + old[:60].strip()
    return text.replace(old, new)


def patch_lkh_findtour(text):
    """FindTour.c of LKH-3.0.14 -> the same file with the recording calls (see _LKH_FUNCTION)."""
    assert sha256_text(text) == LKH_FINDTOUR_SHA256, "FindTour.c is not the file of LKH-3.0.14"
    for old, new in _LKH_EDITS:
        text = _replace_once(text, old, new)
    return text


def patch_eax_env(text, version):
    """env.cpp of GA-EAX (version 'Normal' or 'Large') -> time stamps and deadline (see _EAX_NEW)."""
    assert sha256_text(text) == EAX_ENV_SHA256[version], "env.cpp is not the file of the pinned GA-EAX commit"
    return _EAX_HEAD + _replace_once(text, _EAX_OLD, _EAX_NEW)


def patch_eax_evaluator(text, version):
    """evaluator.cpp of GA-EAX -> the lists of nearest cities come from PilotNearest (EUC_2D, more cities than the
    list length); the original loops stay in place for every other case."""
    assert sha256_text(text) == EAX_EVALUATOR_SHA256[version], "evaluator.cpp is not the file of the pinned GA-EAX commit"
    assert text.startswith(_EAX_EVALUATOR_HEAD)
    if version == "Normal":
        assert text.count(_EAX_NORMAL_START) == 1
        start = text.index(_EAX_NORMAL_START)
        end = text.index("\n}\n", start)
        call = ('  if( strcmp( type, "EUC_2D" ) == 0 && Ncity > fNearNumMax + 1 )\n'
                "    PilotNearest( x, y, Ncity, fNearNumMax, fNearCity, NULL );\n  else\n  {\n")
        text = text[:start] + call + text[start:end] + "\n  }" + text[end:]
    else:
        assert text.count(_EAX_LARGE_START) == 1 and text.count(_EAX_LARGE_END) == 1
        start, end = text.index(_EAX_LARGE_START), text.index(_EAX_LARGE_END)
        call = ("  if( Ncity > fNearNumMax + 1 )\n"
                "    PilotNearest( x, y, Ncity, fNearNumMax, fNearCity, fEdgeDisOrder );\n  else\n  {\n")
        text = text[:start] + call + text[start:end].rstrip("\n") + "\n  }\n\n" + text[end:]
    return _EAX_EVALUATOR_HEAD + _EAX_NEAREST + text[len(_EAX_EVALUATOR_HEAD):]


def source_digest(folder, suffixes=(".cpp", ".h")):
    """One SHA-256 over the names and contents of the source files of a folder."""
    h = hashlib.sha256()
    for name in sorted(os.listdir(folder)):
        if name.endswith(suffixes):
            h.update(name.encode("ascii") + b"\0")
            with open(os.path.join(folder, name), "rb") as f:
                h.update(f.read() + b"\0")
    return h.hexdigest()


# ------------------------------------------------------------------------------------------------------ processes
def run_process(cmd, cwd, env=None, timeout=None):
    """Run a process; on a timeout it is stopped and the output printed so far is kept."""
    full_env = dict(os.environ)
    full_env.update({k: str(v) for k, v in (env or {}).items()})
    t_start = time.time()
    try:
        done = subprocess.run([str(c) for c in cmd], cwd=str(cwd), env=full_env, stdin=subprocess.DEVNULL,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
        out, code, timed_out = done.stdout, done.returncode, False
    except subprocess.TimeoutExpired as exc:
        out, code, timed_out = exc.stdout or b"", None, True
    return dict(stdout=out.decode("latin-1", "replace"), returncode=code, timed_out=timed_out, t_start=t_start,
                wall=time.time() - t_start)


def write_par(path, params):
    with open(path, "w") as f:
        for key, value in params.items():
            f.write(f"{key} = {value}\n")


_PRE_RX = re.compile(r"Preprocessing time = ([0-9.]+) sec")
_TRIAL_RX = re.compile(r"^(\*?)[ \t]*(\d+):[ \t]*Cost[ \t]*=[ \t]*(-?\d+)[^\n]*?Time[ \t]*=[ \t]*([\d.]+)[ \t]*sec", re.M)


def lkh_preprocessing_s(stdout):
    m = _PRE_RX.search(stdout)
    return float(m.group(1)) if m else None


def lkh_trial_costs(stdout):
    """(trial, cost) of every trial line of a TRACE_LEVEL = 2 output (improving trials carry a star)."""
    return [(int(k), int(c)) for _, k, c, _ in _TRIAL_RX.findall(stdout)]


# ------------------------------------------------------------------------------------------------- recorded tours
def record_dtype(n):
    return np.dtype([("kind", "<i8"), ("trial", "<i8"), ("cost", "<i8"), ("lkh_ms", "<i8"), ("epoch_ms", "<i8"),
                     ("ids", "<i4", (int(n),))])


def read_records(path, n):
    """All complete records of a file written by the recording copy of LKH (a last, cut record is dropped)."""
    dtype = record_dtype(n)
    count = os.path.getsize(path) // dtype.itemsize if os.path.exists(path) else 0
    if count == 0:
        return np.zeros(0, dtype=dtype)
    return np.fromfile(path, dtype=dtype, count=count)


def tour_of(record):
    """0-based tour of a record."""
    return record["ids"].astype(np.int64) - 1


def is_permutation(tour, n):
    tour = np.asarray(tour)
    return tour.shape == (n,) and tour.min() == 0 and tour.max() == n - 1 and len(np.unique(tour)) == n


def edge_keys(tour, n):
    """Sorted undirected edge keys of a tour."""
    a = np.asarray(tour, dtype=np.int64)
    b = np.roll(a, -1)
    return np.sort(np.minimum(a, b) * n + np.maximum(a, b))


def improvements(records, start_ms):
    """[[seconds since start_ms, cost], ...] of the new best tours of one process, in time order."""
    best = records[records["kind"] == 1]
    return [[round((int(e) - start_ms) / 1000.0, 3), int(c)] for e, c in zip(best["epoch_ms"], best["cost"])]


def best_before(curve, limit):
    """Smallest cost of a [[t, cost], ...] curve at times <= limit, or None."""
    costs = [c for t, c in curve if t <= limit + 1e-9]
    return min(costs) if costs else None


def select_population(record_list, cut_ms, per_process, n, trial_tours=True):
    """Population at wall-clock time cut_ms: per process its best tour at that time and, with trial_tours, its last
    (per_process - 1) distinct trial tours that ended before that time. Returns 0-based tours, the costs recorded by
    LKH and one information dictionary per process."""
    tours, costs, info = [], [], []
    for records in record_list:
        before = records[records["epoch_ms"] <= cut_ms]
        best = before[before["kind"] == 1]
        trials = before[before["kind"] == 0]
        chosen, seen = [], set()
        if len(best):
            chosen.append(best[-1])
            seen.add(edge_keys(tour_of(best[-1]), n).tobytes())
        used_trials = []
        if trial_tours:
            for i in range(len(trials) - 1, -1, -1):
                if len(chosen) >= per_process:
                    break
                key = edge_keys(tour_of(trials[i]), n).tobytes()
                if key in seen:
                    continue
                seen.add(key)
                chosen.append(trials[i])
                used_trials.append(int(trials[i]["trial"]))
        for rec in chosen:
            tours.append(tour_of(rec))
            costs.append(int(rec["cost"]))
        info.append(dict(trials_before_cut=int(len(trials)), best_tours_before_cut=int(len(best)),
                         best_cost_at_cut=int(best[-1]["cost"]) if len(best) else None,
                         tours_used=int(len(chosen)),
                         trial_range=[min(used_trials), max(used_trials)] if used_trials else None))
    return tours, costs, info


def population_stats(tours, n, pairs=200, seed=0):
    keys = [edge_keys(t, n) for t in tours]
    union, count = np.unique(np.concatenate(keys), return_counts=True)
    rng = np.random.default_rng(seed)
    shared = []
    if len(keys) >= 2:
        for _ in range(min(pairs, len(keys) * (len(keys) - 1) // 2)):
            i, j = rng.choice(len(keys), 2, replace=False)
            shared.append(float(np.isin(keys[i], keys[j], assume_unique=True).mean()))
    return dict(tours=int(len(tours)), distinct=int(len({k.tobytes() for k in keys})),
                union_edges=int(len(union)), common_edges=int((count == len(keys)).sum()),
                pairwise_shared=float(np.mean(shared)) if shared else 1.0)


# ---------------------------------------------------------------------------------------------------------- GA-EAX
def write_eax_problem(path, xy):
    """EUC_2D instance in the layout that GA-EAX reads (keyword, colon and value separated by blanks)."""
    xy = np.asarray(xy)
    integral = np.issubdtype(xy.dtype, np.integer) or bool(np.all(xy == np.round(xy)))
    with open(path, "w") as f:
        f.write(f"NAME : instance\nTYPE : TSP\nDIMENSION : {len(xy)}\nEDGE_WEIGHT_TYPE : EUC_2D\nNODE_COORD_SECTION\n")
        if integral:
            f.writelines(f"{i} {int(x)} {int(y)}\n" for i, (x, y) in enumerate(xy, 1))
        else:
            f.writelines(f"{i} {float(x)!r} {float(y)!r}\n" for i, (x, y) in enumerate(xy, 1))
        f.write("EOF\n")


def write_population(path, tours, costs):
    with open(path, "w") as f:
        for tour, cost in zip(tours, costs):
            f.write(f"{len(tour)} {int(cost)}\n" + " ".join(map(str, (np.asarray(tour) + 1).tolist())) + " \n")


_EAX_LINE = re.compile(r"^(\d+): (-?\d+) ([0-9.eE+-]+) ([0-9.]+) (\d)$", re.M)


def parse_eax_output(stdout, launch_epoch):
    """Generation lines of the patched GA-EAX: improvements as [[seconds since launch, best], ...]."""
    rows = [(int(g), int(b), float(a), float(t), int(s)) for g, b, a, t, s in _EAX_LINE.findall(stdout)]
    curve, best = [], None
    for g, b, a, t, s in rows:
        if best is None or b < best:
            best = b
            curve.append([round(t - launch_epoch, 3), b])
    return dict(generations=rows[-1][0] if rows else None, lines=len(rows), curve=curve,
                first_line_s=round(rows[0][3] - launch_epoch, 3) if rows else None,
                last_line_s=round(rows[-1][3] - launch_epoch, 3) if rows else None,
                last_best=rows[-1][1] if rows else None, last_average=rows[-1][2] if rows else None,
                last_stage=rows[-1][4] if rows else None)


def read_eax_best(path, n):
    """Last tour of a <name>_BestSol file: (0-based tour, length reported by GA-EAX)."""
    with open(path) as f:
        tokens = f.read().split()
    assert len(tokens) >= n + 2 and len(tokens) % (n + 2) == 0, "unexpected GA-EAX tour file"
    block = tokens[-(n + 2):]
    assert int(block[0]) == n
    return np.array(block[2:], dtype=np.int64) - 1, int(block[1])


def run_eax(binary, problem_file, workdir, name, pop_size, kids=30, population_file=None, deadline_epoch=None,
            grace_s=120.0):
    """One GA-EAX run. With population_file the run starts from these tours (pop_size = their number); without it
    GA-EAX builds its own population. deadline_epoch: wall-clock time at which GA-EAX stops with its best tour.
    Returns the parsed output, the best tour (0-based) and the length reported by GA-EAX."""
    os.makedirs(workdir, exist_ok=True)
    for suffix in ("_BestSol", "_Result"):
        path = os.path.join(workdir, name + suffix)
        if os.path.exists(path):
            os.remove(path)
    cmd = [binary, "1", name, str(int(pop_size)), str(int(kids)), problem_file]
    if population_file is not None:
        cmd.append(population_file)
    env = {} if deadline_epoch is None else {"EAX_DEADLINE": f"{deadline_epoch:.3f}"}
    timeout = None if deadline_epoch is None else max(5.0, deadline_epoch - time.time()) + grace_s
    res = run_process(cmd, workdir, env=env, timeout=timeout)
    out = parse_eax_output(res["stdout"], res["t_start"])
    out.update(returncode=res["returncode"], timed_out=res["timed_out"], wall_s=float(res["wall"]),
               launch_epoch=float(res["t_start"]), deadline_s=None if deadline_epoch is None
               else float(deadline_epoch - res["t_start"]), tail=res["stdout"][-400:])
    best_file = os.path.join(workdir, name + "_BestSol")
    out["finished"] = bool(res["returncode"] == 0 and os.path.exists(best_file))
    out["stopped_by_deadline"] = bool(out["finished"] and deadline_epoch is not None and out["last_line_s"] is not None
                                      and res["t_start"] + out["last_line_s"] >= deadline_epoch - 1e-3)
    return out, best_file


# ------------------------------------------------------------------------------------------------------ statistics
def sign_test_p(wins, losses):
    """Exact two-sided sign test (ties excluded)."""
    m = wins + losses
    if m == 0:
        return 1.0
    k = min(wins, losses)
    return float(min(1.0, 2.0 * sum(comb(m, i) for i in range(k + 1)) / 2.0 ** m))


def paired_stats(values, bootstrap_seed, replicates=5000):
    """Paired percentage differences (first minus second arm): wins are negative values (the first arm is shorter)."""
    x = np.asarray(values, dtype=np.float64)
    assert x.ndim == 1 and len(x) and np.isfinite(x).all()
    wins, losses = int((x < -1e-12).sum()), int((x > 1e-12).sum())
    rng = np.random.default_rng(bootstrap_seed)
    means = x[rng.integers(0, len(x), size=(replicates, len(x)))].mean(axis=1)
    low, high = np.quantile(means, [0.025, 0.975])
    return dict(instances=int(len(x)), win=wins, tie=int(len(x) - wins - losses), loss=losses,
                mean_delta_pct=float(x.mean()), median_delta_pct=float(np.median(x)),
                ci95_low_pct=float(low), ci95_high_pct=float(high), sign_p=sign_test_p(wins, losses))


In [ ]:
import hashlib
import importlib.util
import json
import math
import platform
import shutil
import subprocess
import sys
import tarfile
import threading
import time
import traceback
import urllib.request
import uuid
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

if importlib.util.find_spec("pandas") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd

CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
assert CPU_COUNT >= W, f"{W} süreç için en az {W} vCPU gerekli; görünen: {CPU_COUNT}"
if REQUIRE_12_VCPUS:
    assert CPU_COUNT >= 2 * W, f"Önceki koşulardaki düzen için 12 vCPU gerekli (Yüksek RAM seçin); görünen: {CPU_COUNT}"


def cpu_model_name():
    path = Path("/proc/cpuinfo")
    if path.exists():
        for line in path.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"


def gpu_name():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, timeout=30)
        return out.stdout.strip().splitlines()[0] if out.returncode == 0 and out.stdout.strip() else "none"
    except Exception:
        return "none"


CPU_MODEL = cpu_model_name()
GPU_NAME = gpu_name()
WORK_ROOT = Path(os.environ.get("BETATOUR_V25_WORK", "/content/betatour_v25_work"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Depo: önceki koşularla aynı sabit commit (örnek üreticisi ve TSPLIB yardımcıları için; model kullanılmaz).
if TEST_MODE and os.environ.get("BETATOUR_V25_REPO"):
    REPO = Path(os.environ["BETATOUR_V25_REPO"])
else:
    REPO = WORK_ROOT / "repo"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "--quiet",
                        "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
    assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
sys.path.insert(0, str(REPO / "src"))
import tsplab as T

RAM_GB = T.env_info().get("ram_gb")
assert RAM_GB is None or RAM_GB >= MIN_RAM_GB, \
    f"Yüksek RAM gerekli (GA-EAX'in altı paralel süreci için): görünen {RAM_GB} GB, gereken en az {MIN_RAM_GB} GB"

# Bu deneyin yardımcı modülü (önceki hücrenin yazdığı dosya). Dosya sonundaki satır sonu sayısı arayüze göre
# değişebildiği için tek satır sonuna indirilir; içerik depodaki modülle bayt bayt aynı olmalıdır.
CODE_PATH = Path("byprod_v25.py").resolve()
assert CODE_PATH.exists(), "Önce bir önceki hücreyi (byprod_v25.py) çalıştırın."
code_bytes = CODE_PATH.read_bytes().rstrip(b"\n") + b"\n"
CODE_SHA256 = hashlib.sha256(code_bytes).hexdigest()
assert CODE_SHA256 == CODE_SHA256_EXPECTED, "byprod_v25.py beklenen modülle aynı değil; önceki hücreyi değiştirmeden çalıştırın."
if CODE_PATH.read_bytes() != code_bytes:
    CODE_PATH.write_bytes(code_bytes)
sys.path.insert(0, str(CODE_PATH.parent))
import byprod_v25 as B

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / EXPERIMENT_NAME
for sub in ["instances", "reference", "tours", "stages", "errors", "summary", "code"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
shutil.copy2(CODE_PATH, OUTPUT_ROOT / "code" / "byprod_v25.py")


def run_build(cmd, cwd, what):
    build = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=1800)
    if build.returncode != 0:
        raise RuntimeError(what + " derlemesi başarısız:\n" + build.stdout[-3000:])


def findtour_matches(folder):
    path = Path(folder) / "SRC" / "FindTour.c"
    return path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == B.LKH_FINDTOUR_SHA256


def get_lkh_3014():
    """Değiştirilmemiş LKH-3.0.14. Resmi kaynak; ulaşılamazsa ya da FindTour.c beklenen dosya değilse SHA256
    denetimli sabit kopya (run v21-v23 bu kopyayla koştu)."""
    if TEST_MODE and os.environ.get("BETATOUR_V25_LKH"):
        assert findtour_matches(Path(os.environ["BETATOUR_V25_LKH"]).parent), "yerel LKH kaynağı 3.0.14 değil"
        return os.environ["BETATOUR_V25_LKH"], "local test binary"
    root = WORK_ROOT / "lkh"
    binary = root / "LKH-3.0.14" / "LKH"
    marker = root / "source.txt"
    if binary.exists() and marker.exists() and findtour_matches(root / "LKH-3.0.14"):
        return str(binary), marker.read_text().strip()
    root.mkdir(parents=True, exist_ok=True)
    options = [
        ("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("pinned copy", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar",
         "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33", 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout in options:
        archive = root / ("LKH-3.0.14.tar" if url.endswith(".tar") else "LKH-3.0.14.tgz")
        try:
            print("LKH kaynağı:", label, url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, archive.open("wb") as f:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    f.write(block)
            if expected_sha:
                got = hashlib.sha256(archive.read_bytes()).hexdigest()
                assert got == expected_sha, f"LKH kopyası SHA256 uyuşmazlığı: {got}"
            shutil.rmtree(root / "LKH-3.0.14", ignore_errors=True)
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(root, filter="data")
                except TypeError:
                    tar.extractall(root)
            assert (root / "LKH-3.0.14" / "Makefile").exists(), "LKH kaynak klasörü eksik"
            assert findtour_matches(root / "LKH-3.0.14"), "FindTour.c beklenen dosya değil (kayıt yaması uygulanamaz)"
            run_build(["make", "-j", str(min(4, CPU_COUNT))], root / "LKH-3.0.14", "LKH")
            assert binary.exists(), "LKH ikilisi oluşmadı"
            marker.write_text(f"{label}: {url}")
            return str(binary), f"{label}: {url}"
        except Exception as exc:
            last_error = exc
            print(f"{label} başarısız: {exc}")
    raise RuntimeError(f"LKH 3.0.14 indirilemedi veya derlenemedi: {last_error}")


def build_recording_lkh(plain_binary):
    """LKH'nin kayıt eden kopyası: FindTour.c'ye iki çağrı ve bir işlev eklenir (byprod_v25.patch_lkh_findtour).
    Arama durumu değişmez; öz sınama bunu bu makinede denetler."""
    source = Path(plain_binary).parent
    dest = WORK_ROOT / "lkh_rec" / "LKH-3.0.14"
    binary, marker = dest / "LKH", dest / "patched.txt"
    if binary.exists() and marker.exists() and marker.read_text() == B.VERSION:
        return str(binary)
    shutil.rmtree(dest.parent, ignore_errors=True)
    shutil.copytree(source, dest, ignore=shutil.ignore_patterns("OBJ", "LKH", "*.o", "*.tgz", "*.tar"))
    (dest / "SRC" / "OBJ").mkdir(exist_ok=True)
    path = dest / "SRC" / "FindTour.c"
    path.write_text(B.patch_lkh_findtour(path.read_text()))
    run_build(["make", "-j", str(min(4, CPU_COUNT))], dest, "kayıt eden LKH")
    assert binary.exists(), "kayıt eden LKH ikilisi oluşmadı"
    marker.write_text(B.VERSION)
    return str(binary)


def get_eax():
    """GA-EAX (Nagata ve Kobayashi, 2013): yazarın GitHub deposu, sabit commit, MIT lisansı. İki sürüm de
    (Normal, Large) iki yamayla derlenir: env.cpp'ye zaman damgası ve duvar saati sınırı (patch_eax_env),
    evaluator.cpp'ye en yakın şehir listelerinin ızgarayla bulunması (patch_eax_evaluator; aynı listeler, 50.000
    düğümde yaklaşık bir dakika yerine bir saniyeden az). Öz sınama için ikinci yamasız birer kopya da derlenir."""
    if TEST_MODE and os.environ.get("BETATOUR_V25_EAX_SRC"):
        source, label = Path(os.environ["BETATOUR_V25_EAX_SRC"]), "local test clone"
    else:
        source = WORK_ROOT / "ga_eax_src"
        if not (source / ".git").exists():
            shutil.rmtree(source, ignore_errors=True)
            subprocess.run(["git", "clone", "--quiet", "https://github.com/nagata-yuichi/GA-EAX.git", str(source)],
                           check=True)
        subprocess.run(["git", "-C", str(source), "checkout", "--quiet", B.EAX_COMMIT], check=True)
        assert subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip() == B.EAX_COMMIT
        label = "https://github.com/nagata-yuichi/GA-EAX @ " + B.EAX_COMMIT
    binaries, plain = {}, {}
    for version in ("Normal", "Large"):
        folder = source / "GA_EAX_1.0" / version
        assert B.source_digest(str(folder)) == B.EAX_TREE_SHA256[version], \
            f"GA-EAX kaynağı ({version}) sabitlenen commit ile aynı değil"
        for grid, target in ((True, binaries), (False, plain)):
            dest = WORK_ROOT / "ga_eax_build" / (version.lower() + ("" if grid else "_plain"))
            binary, marker = dest / "eax", dest / "patched.txt"
            if not (binary.exists() and marker.exists() and marker.read_text() == B.VERSION):
                shutil.rmtree(dest, ignore_errors=True)
                dest.mkdir(parents=True)
                for path in folder.iterdir():
                    if path.suffix in {".cpp", ".h"}:
                        shutil.copy2(path, dest / path.name)
                env_file = dest / "env.cpp"
                env_file.write_text(B.patch_eax_env(env_file.read_text(), version))
                if grid:
                    evaluator_file = dest / "evaluator.cpp"
                    evaluator_file.write_text(B.patch_eax_evaluator(evaluator_file.read_text(), version))
                run_build(["g++", "-O3", "-w", "-o", "eax"] + B.EAX_SOURCES + ["-lm"], dest, "GA-EAX " + version)
                assert binary.exists(), "GA-EAX ikilisi oluşmadı"
                marker.write_text(B.VERSION)
            target[version] = str(binary)
    return binaries, plain, label


LKH, LKH_SOURCE = get_lkh_3014()
LKH_REC = build_recording_lkh(LKH)
EAX, EAX_PLAIN, EAX_SOURCE = get_eax()

HARDWARE = {
    "gpu": GPU_NAME,
    "vcpus_visible": CPU_COUNT,
    "cpu_model": CPU_MODEL,
    "ram_gb": RAM_GB,
    "python": platform.python_version(),
    "numpy": np.__version__,
}
MANIFEST = {
    "schema": 1,
    "experiment_name": EXPERIMENT_NAME,
    "test_mode": TEST_MODE,
    "design": {"synthetic": [list(x) for x in SYNTHETIC], "tsplib": TSPLIB_NAMES, "candidate_k": K_LIST,
               "splits": [round(float(f), 6) for f in SPLITS], "primary_split": round(float(PRIMARY_SPLIT), 6),
               "per_process": PER_PROCESS, "eax_kids": EAX_KIDS, "eax_scratch_pop": EAX_SCRATCH_POP,
               "eax_normal_max_n": EAX_NORMAL_MAX_N, "reference_max_n": REFERENCE_MAX_N,
               "cascade_tours": CASCADE_TOURS, "cascade_gen_fraction": CASCADE_GEN_FRACTION,
               "cascade_seed0": CASCADE_SEED0, "stop_pad_s": STOP_PAD_S},
    "seeds": SEEDS,
    "workers": W,
    "repo_commit": REPO_COMMIT,
    "code_version": B.VERSION,
    "code_sha256": CODE_SHA256,
    "notebook_build": NOTEBOOK_BUILD,
    "lkh_source": LKH_SOURCE,
    "lkh_findtour_sha256": B.LKH_FINDTOUR_SHA256,
    "eax_source": EAX_SOURCE,
    "eax_commit": B.EAX_COMMIT,
    "hardware": HARDWARE,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in ["schema", "design", "seeds", "workers", "repo_commit", "code_version", "code_sha256",
                "notebook_build", "eax_commit"]:
        assert saved[key] == MANIFEST[key], f"{key} değişti. Ayrı bir EXPERIMENT_NAME kullanın."
    assert saved["hardware"]["vcpus_visible"] == HARDWARE["vcpus_visible"], "vCPU sayısı değişti; aynı ortam türünü seçin."
    same_machine_type = (saved["hardware"]["gpu"] == HARDWARE["gpu"]
                         and saved["hardware"]["cpu_model"] == HARDWARE["cpu_model"])
    assert ALLOW_HARDWARE_CHANGE or same_machine_type, \
        (f"Makine türü değişti ({saved['hardware']['gpu']}, {saved['hardware']['cpu_model']} -> {HARDWARE['gpu']}, "
         f"{HARDWARE['cpu_model']}). Aynı türü seçin; bulunamıyorsa ilk hücrede ALLOW_HARDWARE_CHANGE = True yapın "
         "(her örneğin bütün kolları yine tek makinede koşar).")
    if saved["hardware"] != HARDWARE:
        print("UYARI: donanım veya kütüphane sürümleri ilk oturumdan farklı; her örneğin donanımı ayrıca kaydedilir.")
else:
    manifest_path.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))

# Aynı Colab sanal makinesinde defter yeniden çalıştırılırsa bu işaret korunur. Yeni makinede yarım kalan
# örnek baştan çalıştırılır; böylece bir örneğin bütün kolları tek makinede koşar.
session_path = WORK_ROOT / "runtime_session_id.txt"
if session_path.exists():
    SESSION_ID = session_path.read_text().strip()
else:
    SESSION_ID = uuid.uuid4().hex
    session_path.write_text(SESSION_ID)

print("Hazır:", CPU_COUNT, "vCPU |", RAM_GB, "GB RAM |", CPU_MODEL, "| GPU:", GPU_NAME, "| LKH:", LKH_SOURCE,
      "| GA-EAX:", EAX_SOURCE, "| kod", B.VERSION, CODE_SHA256[:12], "| sonuçlar:", OUTPUT_ROOT)

In [ ]:
def tsplib_budget(n):
    return float(max(10.0, 0.02 * n))


def round_robin(lists):
    out = []
    for i in range(max(len(x) for x in lists)):
        out += [x[i] for x in lists if i < len(x)]
    return out


def make_instances():
    """Yeni sentetik örnekler ve optimumu bilinen TSPLIB örnekleri. Sıra: 10.000 düğümlü örnekler (aileler
    dönüşümlü), TSPLIB örnekleri, en son en büyük örnekler; kısa bir oturum da ilk soruyu yanıtlar."""
    groups = {}
    for group, n, count, family, gen_seed, budget in SYNTHETIC:
        rng = np.random.default_rng(gen_seed)
        groups[group] = [dict(group=group, family=family, index=i, name=f"p25_{group}_{i:03d}",
                              xy=T.to_int_coords(T.gen_instance(n, family, rng)), ew="EUC_2D", problem_file=None,
                              budget=float(budget), optimum=None) for i in range(count)]
    if TEST_MODE:                       # TSPLIB biçimli küçük bir dosya, ondalıklı koordinatlarla; ağ gerekmez
        folder = WORK_ROOT / "tsplib_test"
        folder.mkdir(exist_ok=True)
        paths, optimum = {}, {}
        for name in TSPLIB_NAMES:
            xy = np.random.default_rng(73).random((900, 2)) * 5000.0
            paths[name] = str(folder / (name + ".tsp"))
            with open(paths[name], "w") as f:
                f.write(f"NAME : {name}\nTYPE : TSP\nDIMENSION : 900\nEDGE_WEIGHT_TYPE : EUC_2D\nNODE_COORD_SECTION\n")
                f.writelines(f"{i} {x:.3f} {y:.3f}\n" for i, (x, y) in enumerate(xy, 1))
                f.write("EOF\n")
            optimum[name] = None
    else:
        paths = T.download_tsplib(str(WORK_ROOT / "tsplib"), names=TSPLIB_NAMES)
        absent = sorted(set(TSPLIB_NAMES) - set(paths))
        assert not absent, f"Eksik TSPLIB dosyaları: {absent}. Ağ bağlantısını denetleyip bu hücreyi yeniden çalıştırın."
        optimum = {name: float(T.TSPLIB_OPT[name]) for name in TSPLIB_NAMES}
    groups["tsplib"] = []
    for i, name in enumerate(TSPLIB_NAMES):
        data = T.read_tsplib(paths[name])
        n, ew = int(data["dimension"]), data["edge_weight_type"]
        assert len(data["xy"]) == n and ew == "EUC_2D", (name, n, ew)       # GA-EAX yalnızca EUC_2D okur
        groups["tsplib"].append(dict(group="tsplib", family="tsplib", index=i, name=name, xy=data["xy"], ew=ew,
                                     problem_file=paths[name], budget=10.0 if TEST_MODE else tsplib_budget(n),
                                     optimum=optimum[name]))
    synthetic = [(g, n) for g, n, *_ in SYNTHETIC]
    smallest = min(n for _, n in synthetic)
    first = round_robin([groups[g] for g, n in synthetic if n == smallest])
    last = round_robin([groups[g] for g, n in synthetic if n != smallest] or [[]])
    result = first + groups["tsplib"] + last
    assert len({x["name"] for x in result}) == len(result) == sum(len(v) for v in groups.values())
    return result


INSTANCES = make_instances()


def estimated_seconds(inst):
    """Örnek başına: kayıtlı LKH koşusu (bütçe + ön işleme), GA-EAX birleştirmeleri (en uzunu bütçenin
    1 - en küçük bölme kadarı), bilinen melez (bütçe) ve küçük ek süreler."""
    n, budget = len(inst["xy"]), inst["budget"]
    return (budget + STOP_PAD_S) + (1.0 - min(SPLITS)) * budget + budget + 0.0015 * n + 20.0


def reference_seconds():
    """Sıfırdan GA-EAX: altışar örnek paralel koşar; bir grubun süresi 6 x en uzun bütçedir."""
    pending = [x for x in INSTANCES if len(x["xy"]) <= REFERENCE_MAX_N]
    return sum(W * max(x["budget"] for x in pending[i:i + W]) + 30.0 for i in range(0, len(pending), W))


by_group = {}
for x in INSTANCES:
    by_group[x["group"]] = by_group.get(x["group"], 0.0) + estimated_seconds(x) / 3600.0
estimate_h = sum(by_group.values())
reference_h = reference_seconds() / 3600.0 if RUN_REFERENCE else 0.0
print(f"{len(INSTANCES)} örnek hazır. Tahmini süre: ana koşu yaklaşık {estimate_h:.1f} saat ("
      + ", ".join(f"{g}: {h:.1f} sa" for g, h in by_group.items())
      + f"), sıfırdan GA-EAX referansı yaklaşık {reference_h:.1f} saat; toplam yaklaşık {estimate_h + reference_h:.1f} saat.")

In [ ]:
POP_PARAMS = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_CANDIDATES": K_LIST}
FRACTIONS = [0.25, 0.5, 0.75, 1.0]


def split_tag(f):
    return f"{int(round(100 * f)):02d}"


# Birleştirme kolları: (ad, popülasyon türü, bölme). "trials": her sürecin en iyi turu + son atılan deneme turları;
# "best6": yalnızca altı sürecin en iyi turları (atılan turların katkısını ayıran kontrol).
MERGE_ARMS = [(f"{kind}_{split_tag(f)}", kind, float(f)) for f in SPLITS for kind in ("trials", "best6")]
ARM_NAMES = ["lkh6"] + [name for name, _, _ in MERGE_ARMS] + ["cascade"]
PRIMARY_ARM = "trials_" + split_tag(PRIMARY_SPLIT)
CONTROL_ARM = "best6_" + split_tag(PRIMARY_SPLIT)


def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)


def write_npz_atomic(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)


def parallel(func, items):
    with ThreadPoolExecutor(max_workers=W) as pool:
        return list(pool.map(func, items))


def xy_hash(inst):
    return hashlib.sha256(np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest()


def instance_files(inst):
    """Örneğin yerel diskteki klasörü, LKH'nin okuduğu TSPLIB dosyası ve GA-EAX'in okuduğu dosya."""
    folder = WORK_ROOT / "local" / inst["name"]
    folder.mkdir(parents=True, exist_ok=True)
    lkh_problem = inst["problem_file"] or str(folder / "p.tsp")
    if inst["problem_file"] is None and not Path(lkh_problem).exists():
        T.write_tsp(lkh_problem, inst["xy"], name=inst["name"], ew=inst["ew"])
    eax_problem = folder / "p_eax.tsp"
    if not eax_problem.exists():
        assert inst["ew"] == "EUC_2D"
        B.write_eax_problem(str(eax_problem), inst["xy"])
    return folder, lkh_problem, str(eax_problem)


def eax_version(n):
    return "Normal" if n <= EAX_NORMAL_MAX_N else "Large"


def tour_len(inst, tour):
    return int(T.tour_length(inst["xy"], tour, inst["ew"]))


def checked_eax_tour(inst, out, best_file, label):
    """GA-EAX'in yazdığı tur: geçerli olmalı ve bildirdiği uzunluk kendi hesabımızla aynı olmalı."""
    n = len(inst["xy"])
    assert out["finished"], f"{label}: GA-EAX sonuç yazmadı (çıkış kodu {out['returncode']}): {out['tail']}"
    tour, value = B.read_eax_best(best_file, n)
    assert B.is_permutation(tour, n), f"{label}: GA-EAX turu geçersiz"
    assert tour_len(inst, tour) == value, f"{label}: GA-EAX'in bildirdiği uzunluk turun uzunluğuna eşit değil"
    return tour, value


def eax_record(out, value, version):
    return dict(version=version, generations=out["generations"], converged=bool(not out["stopped_by_deadline"]),
                stopped_by_deadline=bool(out["stopped_by_deadline"]), wall_s=float(out["wall_s"]),
                first_line_s=out["first_line_s"], last_line_s=out["last_line_s"], last_stage=out["last_stage"],
                curve=out["curve"], final_length=int(value), returncode=out["returncode"])


def make_recorded(inst):
    """Altı kayıt eden LKH süreci (POPMUSIC-5, kesintisiz; önceki koşulardaki pop_full kolunun ayarları). Her
    denemenin turu ve her yeni en iyi tur, duvar saati damgasıyla dosyaya yazılır. Süreçler bütçeden STOP_PAD_S
    saniye sonra dışarıdan durdurulur; bütün zamanlar altı sürecin başlatıldığı andan itibaren duvar saatidir."""
    n, budget = len(inst["xy"]), inst["budget"]
    folder, lkh_problem, _ = instance_files(inst)
    rec_dir = folder / "rec"
    shutil.rmtree(rec_dir, ignore_errors=True)
    rec_dir.mkdir(parents=True)
    start = time.time()

    def one(seed):
        par = rec_dir / f"s{seed}.par"
        B.write_par(par, dict(PROBLEM_FILE=lkh_problem, RUNS=1, TRACE_LEVEL=1, TIME_LIMIT=round(1.25 * budget + 30.0, 1),
                              MAX_TRIALS=100000000, SEED=seed, **POP_PARAMS, OUTPUT_TOUR_FILE=rec_dir / f"s{seed}.tour"))
        return B.run_process([LKH_REC, par], rec_dir, env={"BETATOUR_TRIAL_DUMP": rec_dir / f"s{seed}.bin"},
                             timeout=max(1.0, start + budget + STOP_PAD_S - time.time()))

    raw = parallel(one, SEEDS)
    wall = time.time() - start
    start_ms = int(round(start * 1000.0))
    budget_ms = int(math.floor(1000.0 * budget))
    runs, finals = [], []
    for seed, res in zip(SEEDS, raw):
        assert res["timed_out"] or res["returncode"] == 0, f"lkh6 / tohum {seed}: LKH çıkış kodu {res['returncode']}"
        records = B.read_records(rec_dir / f"s{seed}.bin", n)
        within = records[records["epoch_ms"] <= start_ms + budget_ms]
        best = within[within["kind"] == 1]
        assert len(best), f"lkh6 / tohum {seed}: bütçe içinde tur yok"
        tour = B.tour_of(best[-1])
        assert B.is_permutation(tour, n) and tour_len(inst, tour) == int(best[-1]["cost"]), \
            f"lkh6 / tohum {seed}: kaydedilen en iyi tur geçersiz"
        curve = B.improvements(records, start_ms)
        trial_end = (records["epoch_ms"][records["kind"] == 0] - start_ms) / 1000.0
        assert B.best_before(curve, budget_ms / 1000.0) == int(best[-1]["cost"])
        runs.append(dict(seed=int(seed), improvements=curve, best_at_budget=int(best[-1]["cost"]),
                         best_at_fraction=[B.best_before(curve, math.floor(1000.0 * f * budget) / 1000.0)
                                           for f in FRACTIONS],
                         trials_at_fraction=[int((trial_end <= math.floor(1000.0 * f * budget) / 1000.0).sum())
                                             for f in FRACTIONS],
                         pre_time_s=B.lkh_preprocessing_s(res["stdout"]), first_tour_s=curve[0][0],
                         last_record_s=float((int(records["epoch_ms"][-1]) - start_ms) / 1000.0),
                         stopped_from_outside=bool(res["timed_out"]), returncode=res["returncode"],
                         wall_s=float(res["wall"])))
        finals.append((int(best[-1]["cost"]), tour))
    at_fraction = []
    for j in range(len(FRACTIONS)):
        known = [r["best_at_fraction"][j] for r in runs if r["best_at_fraction"][j] is not None]
        at_fraction.append(min(known) if known else None)
    record = dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                  mean_at_budget=float(np.mean([r["best_at_budget"] for r in runs])), best_at_fraction=at_fraction,
                  mean_trials_at_fraction=[float(np.mean([r["trials_at_fraction"][j] for r in runs]))
                                           for j in range(len(FRACTIONS))],
                  wall_s=float(wall), start_ms=start_ms)
    return record, min(finals, key=lambda x: x[0])[1]


def make_merges(inst, rec):
    """Kayıtlı koşunun turlarından popülasyonlar ve GA-EAX. Her bölme için LKH turları bütçenin o kesrindeki
    andan (duvar saati) alınır; GA-EAX'e kalan süre, popülasyonu hazırlama süresi düşülerek verilir. Altı GA-EAX
    süreci (üç bölme x iki popülasyon türü) paralel koşar; her birinin kendi duvar saati sınırı vardır."""
    n, budget = len(inst["xy"]), inst["budget"]
    folder, _, eax_problem = instance_files(inst)
    records = [B.read_records(folder / "rec" / f"s{seed}.bin", n) for seed in SEEDS]
    assert all(len(r) for r in records), "kayıt dosyaları eksik"
    version = eax_version(n)
    jobs = []
    for name, kind, f in MERGE_ARMS:
        cut_ms = int(math.floor(1000.0 * f * budget))
        cut_s = cut_ms / 1000.0
        begin = time.perf_counter()
        tours, costs, info = B.select_population(records, rec["start_ms"] + cut_ms, PER_PROCESS, n,
                                                 trial_tours=(kind == "trials"))
        pop_file = folder / (name + ".pop")
        B.write_population(pop_file, tours, costs)
        prep_s = time.perf_counter() - begin
        known = [B.best_before(run["improvements"], cut_s) for run in rec["runs"]]
        known = [x for x in known if x is not None]
        assert len(tours) >= 2 and known, f"{name}: bütçenin %{100 * f:.0f} anında birleştirilecek tur yok"
        for tour, cost in zip(tours, costs):
            assert B.is_permutation(tour, n) and tour_len(inst, tour) == cost, f"{name}: kaydedilen tur geçersiz"
        assert min(costs) == min(known), f"{name}: popülasyonun en iyi turu altı sürecin en iyi turu değil"
        jobs.append(dict(name=name, kind=kind, split=f, cut_s=float(cut_s), prep_s=float(prep_s), pop_file=pop_file,
                         pop_size=len(tours), best_at_cut=int(min(known)), per_process=info,
                         population=B.population_stats(tours, n)))

    def one(job):
        allowed = max(0.0, budget - job["cut_s"] - job["prep_s"])
        out, best_file = B.run_eax(EAX[version], eax_problem, str(folder / ("eax_" + job["name"])), "M",
                                   job["pop_size"], kids=EAX_KIDS, population_file=str(job["pop_file"]),
                                   deadline_epoch=time.time() + allowed)
        return allowed, out, best_file

    raw = parallel(one, jobs)
    merges, tours_out = {}, {}
    for job, (allowed, out, best_file) in zip(jobs, raw):
        tour, value = checked_eax_tour(inst, out, best_file, job["name"])
        at_deadline = B.best_before(out["curve"], allowed)
        at_budget = min(x for x in (job["best_at_cut"], at_deadline) if x is not None)
        merges[job["name"]] = dict(kind=job["kind"], split=job["split"], cut_s=job["cut_s"], prep_s=job["prep_s"],
                                   eax_budget_s=float(allowed), pop_size=job["pop_size"],
                                   best_at_cut=job["best_at_cut"], population=job["population"],
                                   per_process=job["per_process"], eax=eax_record(out, value, version),
                                   at_budget=int(at_budget))
        tours_out[job["name"]] = tour
        Path(job["pop_file"]).unlink(missing_ok=True)
    return merges, tours_out


def make_cascade(inst):
    """Bilinen melez: altı işçi paralel olarak LKH'nin tek denemelik turlarını üretir (her işçi ilk koşusunda
    POPMUSIC-5 aday dosyasını yazar, sonraki koşularında onu okur), sonra GA-EAX bu turlardan başlar ve bütçenin
    sonunda durur. Tur üretimi bütçenin CASCADE_GEN_FRACTION kesrinde ya da CASCADE_TOURS tura ulaşınca biter."""
    n, budget = len(inst["xy"]), inst["budget"]
    folder, lkh_problem, eax_problem = instance_files(inst)
    cdir = folder / "cascade"
    shutil.rmtree(cdir, ignore_errors=True)
    cdir.mkdir(parents=True)
    start = time.time()
    gen_limit = start + CASCADE_GEN_FRACTION * budget
    lock = threading.Lock()
    state = dict(started=0, failed=0)
    made = []

    def worker(w):
        cand = cdir / f"cand_{w}.txt"
        while True:
            with lock:
                if state["started"] >= CASCADE_TOURS or time.time() >= gen_limit:
                    return
                index = state["started"]
                state["started"] += 1
            par, tour_file = cdir / f"w{w}.par", cdir / f"w{w}.tour"
            tour_file.unlink(missing_ok=True)
            B.write_par(par, dict(PROBLEM_FILE=lkh_problem, RUNS=1, TRACE_LEVEL=1, MAX_TRIALS=1,
                                  SEED=CASCADE_SEED0 + index, **POP_PARAMS, CANDIDATE_FILE=cand,
                                  OUTPUT_TOUR_FILE=tour_file))
            res = B.run_process([LKH, par], cdir, timeout=max(1.0, start + budget - time.time()))
            if res["returncode"] != 0 or not tour_file.exists():
                with lock:
                    state["failed"] += 1
                    give_up = res["timed_out"] or state["failed"] > 3 * W
                if give_up:
                    return
                continue
            tour = T.read_tour(str(tour_file))
            with lock:
                made.append((index, tour, time.time() - start))

    parallel(worker, range(W))
    generation_s = time.time() - start
    made.sort(key=lambda x: x[0])
    tours = [tour for _, tour, _ in made]
    assert len(tours) >= 2, f"cascade: yalnızca {len(tours)} tur üretildi"
    begin = time.perf_counter()
    for tour in tours:
        assert B.is_permutation(tour, n), "cascade: LKH turu geçersiz"
    costs = [tour_len(inst, tour) for tour in tours]
    pop_file = cdir / "cascade.pop"
    B.write_population(pop_file, tours, costs)
    prep_s = time.perf_counter() - begin
    version = eax_version(n)
    eax_start_s = time.time() - start
    allowed = max(0.0, budget - eax_start_s)
    out, best_file = B.run_eax(EAX[version], eax_problem, str(cdir / "eax"), "C", len(tours), kids=EAX_KIDS,
                               population_file=str(pop_file), deadline_epoch=time.time() + allowed)
    tour, value = checked_eax_tour(inst, out, best_file, "cascade")
    at_deadline = B.best_before(out["curve"], allowed)
    at_budget = min(x for x in (min(costs), at_deadline) if x is not None)
    record = dict(tours=len(tours), failed_runs=int(state["failed"]), generation_s=float(generation_s),
                  prep_s=float(prep_s), eax_start_s=float(eax_start_s), eax_budget_s=float(allowed),
                  first_trial_best=int(min(costs)), first_trial_mean=float(np.mean(costs)),
                  tour_done_s=[round(float(t), 3) for _, _, t in made], population=B.population_stats(tours, n),
                  eax=eax_record(out, value, version), at_budget=int(at_budget),
                  wall_s=float(time.time() - start))
    shutil.rmtree(cdir, ignore_errors=True)
    return record, tour


def make_reference(batch):
    """Sıfırdan GA-EAX (yazarın varsayılan ayarları), örnek başına tek süreç; süre sınırı W x bütçe saniyedir
    (altı LKH sürecinin toplam süresi). Gruptaki örnekler paralel koşar (en çok W süreç, LKH kollarındaki yük)."""
    def one(inst):
        n, budget = len(inst["xy"]), inst["budget"]
        folder, _, eax_problem = instance_files(inst)
        version, limit = eax_version(n), W * budget
        out, best_file = B.run_eax(EAX[version], eax_problem, str(folder / "eax_scratch"), "R", EAX_SCRATCH_POP,
                                   kids=EAX_KIDS, deadline_epoch=time.time() + limit)
        tour, value = checked_eax_tour(inst, out, best_file, inst["name"] + " / referans")
        record = dict(schema=1, complete=True, name=inst["name"], group=inst["group"], n=n, budget_s=float(budget),
                      limit_s=float(limit), population=EAX_SCRATCH_POP, kids=EAX_KIDS, xy_sha256=xy_hash(inst),
                      eax=eax_record(out, value, version), at_limit=B.best_before(out["curve"], limit),
                      at_budget_wall=B.best_before(out["curve"], budget), parallel_runs=len(batch),
                      hardware=HARDWARE, session_id=SESSION_ID, code_sha256=CODE_SHA256,
                      finished_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        shutil.rmtree(folder / "eax_scratch", ignore_errors=True)
        return record, tour

    return parallel(one, batch)


def run_one_instance(inst, ordinal):
    name, n = inst["name"], len(inst["xy"])
    final_path = OUTPUT_ROOT / "instances" / (name + ".json")
    stage_dir = OUTPUT_ROOT / "stages" / name
    local = WORK_ROOT / "local" / name
    session_marker = stage_dir / "runtime_session_id.txt"
    if stage_dir.exists() and any(stage_dir.iterdir()):
        old_id = session_marker.read_text().strip() if session_marker.exists() else None
        if old_id != SESSION_ID:
            print("Yeni Colab makinesi: yarım kalan örnek baştan çalıştırılıyor:", name, flush=True)
            shutil.rmtree(stage_dir)
            shutil.rmtree(local, ignore_errors=True)
    stage_dir.mkdir(parents=True, exist_ok=True)
    session_marker.write_text(SESSION_ID)
    rec_path, merge_path, cascade_path = stage_dir / "lkh6.json", stage_dir / "merges.json", stage_dir / "cascade.json"
    # Kayıtlı LKH koşusu (ve birleştirmeleri) ile bilinen melezin sırası, aile içi sıra numarasına göre dönüşümlüdür.
    lkh_first = bool(inst["index"] % 2 == 0)

    def ensure_lkh_and_merges():
        if rec_path.exists() and merge_path.exists():
            return
        dumps = all((local / "rec" / f"s{seed}.bin").exists() for seed in SEEDS)
        if not (rec_path.exists() and dumps):
            print(name, "/ altı kayıtlı LKH süreci", flush=True)
            rec, tour = make_recorded(inst)
            write_npz_atomic(stage_dir / "tour_lkh6.npz", lkh6=tour.astype(np.int32))
            write_json_atomic(rec_path, rec)
        rec = json.loads(rec_path.read_text())
        print(name, "/ GA-EAX birleştirmeleri (" + ", ".join(a for a, _, _ in MERGE_ARMS) + ")", flush=True)
        merges, tours = make_merges(inst, rec)
        write_npz_atomic(stage_dir / "tours_merges.npz", **{k: v.astype(np.int32) for k, v in tours.items()})
        write_json_atomic(merge_path, merges)
        shutil.rmtree(local / "rec", ignore_errors=True)

    def ensure_cascade():
        if cascade_path.exists():
            return
        print(name, "/ cascade (bilinen melez)", flush=True)
        record, tour = make_cascade(inst)
        write_npz_atomic(stage_dir / "tour_cascade.npz", cascade=tour.astype(np.int32))
        write_json_atomic(cascade_path, record)

    for step in ([ensure_lkh_and_merges, ensure_cascade] if lkh_first else [ensure_cascade, ensure_lkh_and_merges]):
        step()
    rec, merges, cascade = (json.loads(p.read_text()) for p in (rec_path, merge_path, cascade_path))
    results = {"lkh6": int(rec["best_at_budget"]), "cascade": int(cascade["at_budget"])}
    results.update({arm: int(merges[arm]["at_budget"]) for arm, _, _ in MERGE_ARMS})
    assert set(results) == set(ARM_NAMES)
    assert all(merges[arm]["best_at_cut"] >= rec["best_at_budget"] for arm, _, _ in MERGE_ARMS)
    tours = {}
    for file in ("tour_lkh6.npz", "tours_merges.npz", "tour_cascade.npz"):
        with np.load(stage_dir / file, allow_pickle=False) as z:
            tours.update({k: z[k] for k in z.files})
    assert set(tours) == set(ARM_NAMES)
    final_lengths = {arm: tour_len(inst, tours[arm].astype(np.int64)) for arm in ARM_NAMES}
    assert all(final_lengths[arm] <= results[arm] for arm in ARM_NAMES), "kaydedilen tur, kaydedilen sonuçtan uzun"
    write_npz_atomic(OUTPUT_ROOT / "tours" / (name + ".npz"), **tours)
    data = dict(schema=1, complete=True, name=name, group=inst["group"], family=inst["family"],
                index=int(inst["index"]), ordinal=int(ordinal), n=n, budget_s=float(inst["budget"]), ew=inst["ew"],
                optimum=inst["optimum"], xy_sha256=xy_hash(inst), seeds=SEEDS, fractions=FRACTIONS,
                lkh_first=lkh_first, hardware=HARDWARE, session_id=SESSION_ID, code_sha256=CODE_SHA256,
                results=results, final_tour_lengths=final_lengths, lkh6=rec, merges=merges, cascade=cascade,
                finished_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    write_json_atomic(final_path, data)
    shutil.rmtree(local, ignore_errors=True)
    print("TAMAM", name, "|", ", ".join(f"{arm}={results[arm]}" for arm in ARM_NAMES), flush=True)
    return data

In [ ]:
# Öz sınama (ölçüme dahil değildir). Bu makinede ve derlenen ikililerle:
#   1) Kayıt eden LKH, değiştirilmemiş LKH ile aynı aramayı yapıyor mu? Sabit deneme sayılı (belirlenimci) iki koşu
#      her denemede aynı maliyeti vermelidir.
#   2) Kaydedilen her tur geçerli mi ve uzunluğu kaydedilen maliyete eşit mi? Son "en iyi tur" kaydı, LKH'nin
#      kendi yazdığı tur mu?
#   3) GA-EAX kaydedilen turlardan başlayınca geçerli bir tur veriyor mu, bildirdiği uzunluk kendi hesabımızla aynı
#      mı, en iyi girdi turundan uzun olmayan bir tur mu? İki sürüm de (Normal, Large) denenir.
#   4) GA-EAX duvar saati sınırında en iyi turuyla duruyor mu?
#   5) En yakın şehir listelerini ızgarayla bulan GA-EAX, bu yama olmadan derlenen GA-EAX ile aynı turu, aynı kuşak
#      sayısında veriyor mu?
def selfcheck(n=2000, trials=25):
    xy = T.to_int_coords(T.gen_instance(n, "clustered", np.random.default_rng(9093)))
    folder = WORK_ROOT / "selfcheck"
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    problem, eax_problem = folder / "p.tsp", folder / "p_eax.tsp"
    T.write_tsp(str(problem), xy, name="selfcheck", ew="EUC_2D")
    B.write_eax_problem(str(eax_problem), xy)

    def lkh(binary, tag, dump):
        par = folder / (tag + ".par")
        B.write_par(par, dict(PROBLEM_FILE=problem, RUNS=1, TRACE_LEVEL=2, MAX_TRIALS=trials, SEED=SEEDS[0],
                              **POP_PARAMS, OUTPUT_TOUR_FILE=folder / (tag + ".tour")))
        res = B.run_process([binary, par], folder, env={"BETATOUR_TRIAL_DUMP": dump} if dump else None, timeout=900)
        assert res["returncode"] == 0, res["stdout"][-500:]
        return res

    plain, recorded = lkh(LKH, "plain", None), lkh(LKH_REC, "rec", folder / "rec.bin")
    costs_plain, costs_rec = B.lkh_trial_costs(plain["stdout"]), B.lkh_trial_costs(recorded["stdout"])
    records = B.read_records(folder / "rec.bin", n)
    trial_rec, best_rec = records[records["kind"] == 0], records[records["kind"] == 1]
    valid = all(B.is_permutation(B.tour_of(r), n) and int(T.tour_length(xy, B.tour_of(r))) == int(r["cost"])
                for r in records)
    own = T.read_tour(str(folder / "rec.tour"))
    tours, costs, _ = B.select_population([records], int(records["epoch_ms"][-1]), 12, n)
    B.write_population(folder / "pop", tours, costs)
    eax = {}
    for version in ("Normal", "Large"):
        out, best_file = B.run_eax(EAX[version], str(eax_problem), str(folder / ("eax_" + version)), "S", len(tours),
                                   kids=EAX_KIDS, population_file=str(folder / "pop"), deadline_epoch=time.time() + 120)
        tour, value = B.read_eax_best(best_file, n)
        ref, ref_file = B.run_eax(EAX_PLAIN[version], str(eax_problem), str(folder / ("plain_" + version)), "S",
                                  len(tours), kids=EAX_KIDS, population_file=str(folder / "pop"),
                                  deadline_epoch=time.time() + 120)
        ref_tour, ref_value = B.read_eax_best(ref_file, n)
        eax[version] = dict(finished=bool(out["finished"]), valid=bool(B.is_permutation(tour, n)),
                            length_reported=int(value), length_recomputed=int(T.tour_length(xy, tour)),
                            generations=out["generations"], wall_s=float(out["wall_s"]),
                            setup_s=out["first_line_s"], setup_without_grid_s=ref["first_line_s"],
                            same_as_without_grid=bool(ref_value == value and ref["generations"] == out["generations"]
                                                      and np.array_equal(ref_tour, tour)))
    # Sınır şimdiki an: GA-EAX popülasyonunu kurar, ilk denetimde durur ve o andaki en iyi turunu yazar.
    out, best_file = B.run_eax(EAX["Normal"], str(eax_problem), str(folder / "eax_deadline"), "D", EAX_SCRATCH_POP,
                               kids=EAX_KIDS, deadline_epoch=time.time())
    tour, value = B.read_eax_best(best_file, n)
    result = dict(
        nodes=n, trials=trials,
        same_search=bool(len(costs_plain) == trials and costs_plain == costs_rec),
        trial_tours_recorded=int(len(trial_rec)), best_tours_recorded=int(len(best_rec)),
        recorded_tours_valid=bool(valid and list(trial_rec["trial"]) == list(range(1, trials + 1))),
        last_best_is_lkh_tour=bool(np.array_equal(B.edge_keys(own, n), B.edge_keys(B.tour_of(best_rec[-1]), n))
                                   and int(best_rec[-1]["cost"]) == min(c for _, c in costs_rec)),
        wall_plain_s=float(plain["wall"]), wall_recording_s=float(recorded["wall"]),
        best_input=int(min(costs)), eax=eax,
        deadline=dict(finished=bool(out["finished"]), stopped_by_deadline=bool(out["stopped_by_deadline"]),
                      valid=bool(B.is_permutation(tour, n) and int(T.tour_length(xy, tour)) == int(value)),
                      wall_s=float(out["wall_s"])))
    shutil.rmtree(folder, ignore_errors=True)
    return result


SELFCHECK = selfcheck(n=600, trials=12) if TEST_MODE else selfcheck()
SELFCHECK["session_id"] = SESSION_ID
write_json_atomic(OUTPUT_ROOT / "summary" / "selfcheck.json", SELFCHECK)
assert SELFCHECK["same_search"], "Kayıt eden LKH, değiştirilmemiş LKH ile aynı aramayı yapmadı."
assert SELFCHECK["recorded_tours_valid"] and SELFCHECK["last_best_is_lkh_tour"], "Kaydedilen turlar beklenen gibi değil."
for version, item in SELFCHECK["eax"].items():
    assert item["finished"] and item["valid"] and item["length_reported"] == item["length_recomputed"], \
        f"GA-EAX ({version}) geçerli ve uzunluğu doğru bir tur vermedi."
    assert item["length_reported"] <= SELFCHECK["best_input"], f"GA-EAX ({version}) en iyi girdi turundan uzun tur verdi."
    assert item["same_as_without_grid"], f"GA-EAX ({version}): ızgaralı komşu araması yamasız derlemeyle aynı sonucu vermedi."
assert SELFCHECK["deadline"]["finished"] and SELFCHECK["deadline"]["stopped_by_deadline"] and SELFCHECK["deadline"]["valid"], \
    "GA-EAX duvar saati sınırında en iyi turuyla durmadı."
print("Öz sınama tamam. Kayıt eden LKH aynı aramayı yaptı (" + str(SELFCHECK["trials"]) + " deneme; süre "
      f"{SELFCHECK['wall_plain_s']:.1f} s / kayıtla {SELFCHECK['wall_recording_s']:.1f} s). GA-EAX: "
      + ", ".join(f"{v} {x['length_reported']} ({x['generations']} kuşak)" for v, x in SELFCHECK["eax"].items())
      + f"; en iyi girdi turu {SELFCHECK['best_input']}.")

In [ ]:
REFERENCE_INSTANCES = [x for x in INSTANCES if len(x["xy"]) <= REFERENCE_MAX_N] if RUN_REFERENCE else []


def save_progress(last_instance=None, state="running"):
    finished = sorted(p.stem for p in (OUTPUT_ROOT / "instances").glob("*.json"))
    reference = sorted(p.stem for p in (OUTPUT_ROOT / "reference").glob("*.json"))
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      dict(state=state, completed=len(finished), total=len(INSTANCES), finished=finished,
                           reference_completed=len(reference), reference_total=len(REFERENCE_INSTANCES),
                           last_instance=last_instance,
                           updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())))


save_progress(state="running")
session_start = time.time()
newly_finished = 0
for ordinal, inst in enumerate(INSTANCES):
    final_file = OUTPUT_ROOT / "instances" / (inst["name"] + ".json")
    if final_file.exists():
        record = json.loads(final_file.read_text())
        assert record.get("complete") and record.get("xy_sha256") == xy_hash(inst) \
            and set(record["results"]) == set(ARM_NAMES), f"{inst['name']}: eski kayıt bu kurulumla uyuşmuyor"
        print("ATLA (tamam):", inst["name"], flush=True)
        continue
    if MAX_NEW_INSTANCES is not None and newly_finished >= MAX_NEW_INSTANCES:
        print("Bu oturum için MAX_NEW_INSTANCES sınırına ulaşıldı.")
        break
    try:
        run_one_instance(inst, ordinal)
        newly_finished += 1
        save_progress(last_instance=inst["name"])
        done_now = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
        left = sum(estimated_seconds(x) for x in INSTANCES
                   if not (OUTPUT_ROOT / "instances" / (x["name"] + ".json")).exists())
        print(f"  {done_now}/{len(INSTANCES)} örnek bitti; bu oturumda geçen süre "
              f"{(time.time() - session_start) / 60:.0f} dk, kalan ana koşu yaklaşık {left / 3600:.1f} saat.", flush=True)
    except Exception as exc:
        failure = dict(instance=inst["name"], error=repr(exc), traceback=traceback.format_exc(),
                       utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        write_json_atomic(OUTPUT_ROOT / "errors" / (inst["name"] + ".json"), failure)
        save_progress(last_instance=inst["name"], state="error")
        raise

done = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
print(f"Bu oturumda yeni {newly_finished}, toplam {done}/{len(INSTANCES)} örnek tamamlandı. "
      f"Süre: {(time.time() - session_start) / 3600:.2f} saat. Kayıt: {OUTPUT_ROOT}")

# Sıfırdan GA-EAX referansı: ana örneklerin hepsi bittikten sonra, altışar örnek paralel.
if done == len(INSTANCES) and REFERENCE_INSTANCES:
    pending = []
    for inst in REFERENCE_INSTANCES:
        path = OUTPUT_ROOT / "reference" / (inst["name"] + ".json")
        if path.exists():
            assert json.loads(path.read_text()).get("xy_sha256") == xy_hash(inst), f"{inst['name']}: eski referans kaydı uyuşmuyor"
        else:
            pending.append(inst)
    for i in range(0, len(pending), W):
        batch = pending[i:i + W]
        print("Sıfırdan GA-EAX referansı:", ", ".join(x["name"] for x in batch),
              f"(yaklaşık {W * max(x['budget'] for x in batch) / 60:.0f} dk)", flush=True)
        try:
            for record, tour in make_reference(batch):
                write_npz_atomic(OUTPUT_ROOT / "tours" / (record["name"] + "__eax_scratch.npz"),
                                 eax_scratch=tour.astype(np.int32))
                write_json_atomic(OUTPUT_ROOT / "reference" / (record["name"] + ".json"), record)
            save_progress(last_instance=batch[-1]["name"])
        except Exception as exc:
            failure = dict(instance="reference: " + ", ".join(x["name"] for x in batch), error=repr(exc),
                           traceback=traceback.format_exc(), utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
            write_json_atomic(OUTPUT_ROOT / "errors" / ("reference_" + batch[0]["name"] + ".json"), failure)
            save_progress(last_instance=batch[0]["name"], state="error")
            raise

reference_done = len(list((OUTPUT_ROOT / "reference").glob("*.json")))
complete = done == len(INSTANCES) and reference_done >= len(REFERENCE_INSTANCES)
save_progress(state="complete" if complete else "partial")
print(f"Ana koşu {done}/{len(INSTANCES)}, referans {reference_done}/{len(REFERENCE_INSTANCES)}. "
      f"Oturum süresi: {(time.time() - session_start) / 3600:.2f} saat.")

In [ ]:
# (anahtar, ilk kol, ikinci kol, açıklama). Yüzde fark = 100 x (ilk - ikinci) / ikinci; negatif: ilk kol daha kısa.
# "eax_scratch" bir kol değil referanstır: tek süreç, W x bütçe saniye (altı LKH sürecinin toplam süresi).
COMPARISONS = [
    ("P1", PRIMARY_ARM, "lkh6", "Atılan turlar + GA-EAX / altı LKH süreci"),
    ("C1", CONTROL_ARM, "lkh6", "Yalnızca altı en iyi tur + GA-EAX / altı LKH süreci"),
    ("C2", PRIMARY_ARM, CONTROL_ARM, "Atılan turların katkısı: atılan turlar / yalnızca altı en iyi tur"),
    ("C3", "cascade", "lkh6", "Bilinen melez (tek denemelik turlar + GA-EAX) / altı LKH süreci"),
    ("C4", PRIMARY_ARM, "cascade", "Atılan turlar / bilinen melez"),
] + [(f"S{i}", arm, "lkh6", f"Diğer bölme: {arm} / altı LKH süreci")
     for i, arm in enumerate([a for a, _, _ in MERGE_ARMS if a not in (PRIMARY_ARM, CONTROL_ARM)], 1)] + [
    ("R1", "eax_scratch", "lkh6", "Sıfırdan GA-EAX (eşit toplam CPU süresi) / altı LKH süreci"),
    ("R2", PRIMARY_ARM, "eax_scratch", "Atılan turlar / sıfırdan GA-EAX (eşit toplam CPU süresi)"),
]


def pct(a, b):
    return 100.0 * (a - b) / b


def size_class(item):
    return "tsplib" if item["group"] == "tsplib" else f"n{round(item['n'] / 1000)}k"


def load_finished():
    by_name = {x["name"]: x for x in INSTANCES}
    found = []
    for path in sorted((OUTPUT_ROOT / "instances").glob("*.json")):
        item = json.loads(path.read_text())
        if item.get("complete") and item["name"] in by_name:
            assert set(item["results"]) == set(ARM_NAMES) and item["xy_sha256"] == xy_hash(by_name[item["name"]])
            ref_path = OUTPUT_ROOT / "reference" / (item["name"] + ".json")
            item["reference"] = json.loads(ref_path.read_text()) if ref_path.exists() else None
            found.append(item)
    return sorted(found, key=lambda r: r["ordinal"])


def match_time(item, arm):
    """Kolun, altı LKH sürecinin bütçe sonundaki uzunluğuna ulaştığı an (ortak saat, saniye); ulaşmadıysa None."""
    target, merge = item["results"]["lkh6"], item["merges"][arm]
    if merge["best_at_cut"] <= target:
        return merge["cut_s"]
    for t, cost in merge["eax"]["curve"]:
        if cost <= target and t <= merge["eax_budget_s"] + 1e-9:
            return merge["cut_s"] + merge["prep_s"] + t
    return None


records = load_finished()
rows = []
for item in records:
    primary, cascade, ref = item["merges"][PRIMARY_ARM], item["cascade"], item["reference"]
    values = dict(item["results"])
    values["eax_scratch"] = ref["at_limit"] if ref and ref["at_limit"] is not None else None
    reached = match_time(item, PRIMARY_ARM)
    row = dict(instance=item["name"], group=item["group"], size_class=size_class(item), n=item["n"],
               budget_s=item["budget_s"], optimum=item["optimum"], ordinal=item["ordinal"],
               lkh_first=item["lkh_first"], cpu_model=item["hardware"]["cpu_model"], gpu=item["hardware"]["gpu"],
               session=item["session_id"][:8])
    for arm in ARM_NAMES + ["eax_scratch"]:
        row[arm] = values[arm]
        if item["optimum"] and values[arm] is not None:
            row[arm + "_gap_pct"] = pct(values[arm], item["optimum"])
    for key, first, second, _ in COMPARISONS:
        a, b = values[first], values[second]
        row[key] = pct(a, b) if a is not None and b is not None else float("nan")
    row.update(
        lkh6_mean6=item["lkh6"]["mean_at_budget"], lkh6_trials=item["lkh6"]["mean_trials_at_fraction"][-1],
        lkh6_stopped_from_outside=int(sum(r["stopped_from_outside"] for r in item["lkh6"]["runs"])),
        lkh6_latest_first_tour_s=max(r["first_tour_s"] for r in item["lkh6"]["runs"]),
        primary_pop_size=primary["pop_size"],
        primary_trials_at_cut=float(np.mean([p["trials_before_cut"] for p in primary["per_process"]])),
        primary_union_per_node=primary["population"]["union_edges"] / item["n"],
        primary_common_per_node=primary["population"]["common_edges"] / item["n"],
        primary_pairwise_shared=primary["population"]["pairwise_shared"], primary_prep_s=primary["prep_s"],
        primary_eax_budget_s=primary["eax_budget_s"], primary_eax_wall_s=primary["eax"]["wall_s"],
        primary_eax_generations=primary["eax"]["generations"], primary_eax_converged=primary["eax"]["converged"],
        primary_match_time_s=reached if reached is not None else float("nan"),
        primary_match_time_share=reached / item["budget_s"] if reached is not None else float("nan"),
        cascade_tours=cascade["tours"], cascade_generation_s=cascade["generation_s"],
        cascade_eax_budget_s=cascade["eax_budget_s"], cascade_eax_converged=cascade["eax"]["converged"],
        cascade_pairwise_shared=cascade["population"]["pairwise_shared"],
        reference_converged=ref["eax"]["converged"] if ref else None,
        reference_wall_s=ref["eax"]["wall_s"] if ref else float("nan"),
        reference_init_s=ref["eax"]["first_line_s"] if ref else float("nan"))
    rows.append(row)

per_instance = pd.DataFrame(rows)
per_instance.to_csv(OUTPUT_ROOT / "summary" / "per_instance.csv", index=False)

stat_rows = []
if len(per_instance):
    frames = [("all", per_instance)] + [("class:" + k, f) for k, f in per_instance.groupby("size_class", sort=True)] \
        + [("group:" + k, f) for k, f in per_instance.groupby("group", sort=True)]
    for group, frame in frames:
        for ci, (key, first, second, label) in enumerate(COMPARISONS):
            values = frame[key].dropna().to_numpy()
            if len(values):
                stat_rows.append(dict(group=group, key=key, comparison=label, first=first, second=second,
                                      **B.paired_stats(values, 20261125 + 100 * ci)))
stats = pd.DataFrame(stat_rows)
stats.to_csv(OUTPUT_ROOT / "summary" / "paired_stats.csv", index=False)

suite_complete = len(INSTANCES) > 0 and len(records) == len(INSTANCES)
if TEST_MODE:
    note = "TEKNİK ÖZ SINAMA: bu sonuç hiçbir karar için kullanılmaz."
elif not suite_complete:
    note = f"ARA SONUÇ: {len(records)}/{len(INSTANCES)} örnek tamamlandı. Karar, koşu bitince verilir."
else:
    note = f"{len(records)}/{len(INSTANCES)} örnek tamamlandı."


def stat(group, key):
    hit = stats[(stats["group"] == group) & (stats["key"] == key)] if len(stats) else stats
    return hit.iloc[0] if len(hit) else None


def describe(row):
    return (f"{row['win']}/{row['tie']}/{row['loss']} (ilk kol kısa/eşit/uzun), ortalama %{row['mean_delta_pct']:+.4f} "
            f"(%95 GA {row['ci95_low_pct']:+.4f} ile {row['ci95_high_pct']:+.4f}), ortanca %{row['median_delta_pct']:+.4f}, "
            f"işaret testi p = {row['sign_p']:.4f}")


# Önceden yazılan karar kuralı (PROTOCOL_v25.md). Pilot keşif amaçlıdır; kural yalnızca bir doğrulama koşusunun
# hazırlanıp hazırlanmayacağını belirler.
classes = sorted(per_instance["size_class"].unique(), key=lambda c: (c == "tsplib", c)) if len(per_instance) else []
synthetic_classes = sorted([c for c in classes if c != "tsplib"], key=lambda c: int(c[1:-1]))
small, large = (synthetic_classes[0], synthetic_classes[-1]) if synthetic_classes else (None, None)
decision = {}
if suite_complete and not TEST_MODE and small and large and small != large:
    d1, d2, d3 = stat("class:" + small, "P1"), stat("class:" + large, "P1"), stat("class:" + small, "C2")
    decision["D1_replication_small"] = bool(d1["win"] > d1["loss"] and d1["sign_p"] < 0.05)
    decision["D2_growth_with_size"] = bool(d2["loss"] == 0 and d2["tie"] == 0 and d2["mean_delta_pct"] < d1["mean_delta_pct"])
    decision["D3_discarded_tours_matter"] = bool(d3["win"] > d3["loss"] and d3["sign_p"] < 0.05)
    decision["confirmation_run"] = bool(decision["D1_replication_small"] and decision["D2_growth_with_size"])

lines = ["# BetaTour run v25 (pilot): atılan deneme turlarının birleştirilmesi", "", note, "",
         f"Birincil kol: `{PRIMARY_ARM}` (altı LKH süreci bütçenin yarısında durur; her sürecin en iyi turu ve son "
         f"{PER_PROCESS - 1} atılan deneme turu GA-EAX ile birleştirilir; toplam süre bütçeye eşittir). Karşılaştırma: "
         "`lkh6` (aynı altı süreç bütçenin sonuna kadar sürer; altısının en iyisi). Bütün zamanlar duvar saatidir.", ""]
lines += ["## Bütçe sonunda karşılaştırmalar", "",
          "| Grup | Karşılaştırma | Örnek | Kısa/eşit/uzun | Ortalama fark (%) | %95 GA | Ortanca (%) | p |", "|---|---|---|---|---|---|---|---|"]
for group in ["all"] + ["class:" + c for c in classes]:
    for key, first, second, label in COMPARISONS:
        row = stat(group, key)
        if row is not None:
            lines.append(f"| {group.replace('class:', '')} | {key}: `{first}` / `{second}` | {row['instances']} | "
                         f"{row['win']}/{row['tie']}/{row['loss']} | {row['mean_delta_pct']:+.4f} | "
                         f"{row['ci95_low_pct']:+.4f} ile {row['ci95_high_pct']:+.4f} | {row['median_delta_pct']:+.4f} | "
                         f"{row['sign_p']:.4f} |")
with_optimum = per_instance[per_instance["optimum"].notna()] if len(per_instance) else per_instance
if len(with_optimum):
    shown = ["lkh6", CONTROL_ARM, PRIMARY_ARM, "cascade", "eax_scratch"]
    lines += ["", "## Optimumu bilinen örneklerde optimuma uzaklık (%)", "",
              "| Örnek | Düğüm | Bütçe (s) | " + " | ".join(f"`{a}`" for a in shown) + " |", "|---|---|---|" + "---|" * len(shown)]
    for _, r in with_optimum.iterrows():
        lines.append(f"| {r['instance']} | {r['n']} | {r['budget_s']:.0f} | "
                     + " | ".join("-" if pd.isna(r.get(a + "_gap_pct", float("nan"))) else f"{r[a + '_gap_pct']:.4f}"
                                  for a in shown) + " |")
    lines.append("| ortalama | | | " + " | ".join(
        "-" if with_optimum[a + "_gap_pct"].isna().any() else f"{with_optimum[a + '_gap_pct'].mean():.4f}"
        for a in shown if a + "_gap_pct" in with_optimum) + " |")
if len(per_instance):
    lines += ["", "## Mekanizma (birincil kol, sınıf ortalamaları)", "",
              "| Sınıf | Örnek | Kesimde deneme/süreç | Birleşim kenarı/düğüm | Ortak kenar/düğüm | İkili ortak kenar payı | "
              "GA-EAX süresi (s) | Yakınsayan | LKH'nin son uzunluğuna ulaşma (bütçenin kesri) | Bilinen melezde tur sayısı |",
              "|---|---|---|---|---|---|---|---|---|---|"]
    for c in classes:
        f = per_instance[per_instance["size_class"] == c]
        share = f["primary_match_time_share"].dropna()
        lines.append(f"| {c} | {len(f)} | {f['primary_trials_at_cut'].mean():.0f} | {f['primary_union_per_node'].mean():.3f} | "
                     f"{f['primary_common_per_node'].mean():.3f} | {f['primary_pairwise_shared'].mean():.3f} | "
                     f"{f['primary_eax_wall_s'].mean():.0f} | {int(f['primary_eax_converged'].sum())}/{len(f)} | "
                     + (f"{share.mean():.2f} ({len(share)}/{len(f)} örnek)" if len(share) else "ulaşmadı")
                     + f" | {f['cascade_tours'].mean():.0f} |")
lines += ["", "## Karar kuralı (koşudan önce yazıldı)", ""]
if decision:
    lines += [f"- D1, küçük boyutta tekrar ({small}): `{PRIMARY_ARM}` / `lkh6` daha çok örnekte kısa ve p < 0,05: "
              f"**{'evet' if decision['D1_replication_small'] else 'hayır'}** ({describe(stat('class:' + small, 'P1'))}).",
              f"- D2, kazanç boyutla büyüyor ({large}): bütün örneklerde kısa ve ortalama fark {small} sınıfındakinden büyük: "
              f"**{'evet' if decision['D2_growth_with_size'] else 'hayır'}** ({describe(stat('class:' + large, 'P1'))}).",
              f"- D3, atılan turların katkısı ({small}): `{PRIMARY_ARM}` / `{CONTROL_ARM}` daha çok örnekte kısa ve p < 0,05: "
              f"**{'evet' if decision['D3_discarded_tours_matter'] else 'hayır'}** ({describe(stat('class:' + small, 'C2'))}).",
              "", "**Sonuç:** " + ("D1 ve D2 sağlandı: doğrulama koşusu hazırlanır (yeni örnekler, önceden gönderilen protokol)."
                                   if decision["confirmation_run"] else
                                   "D1 ve D2 birlikte sağlanmadı: doğrulama koşusu hazırlanmaz; sonuç depoya kısa bir kayıt olarak yazılır.")]
else:
    lines.append("Koşu tamamlanınca değerlendirilir.")
flags = []
if len(per_instance):
    if (per_instance["lkh6_stopped_from_outside"] < W).any():
        flags.append("Bazı LKH süreçleri bütçeden önce kendiliğinden bitti: "
                     + ", ".join(per_instance.loc[per_instance["lkh6_stopped_from_outside"] < W, "instance"]))
    late = per_instance[per_instance["lkh6_latest_first_tour_s"] > min(SPLITS) * per_instance["budget_s"]]
    if len(late):
        flags.append("İlk bölmeden sonra ilk turunu bitiren süreç var: " + ", ".join(late["instance"]))
    if per_instance["session"].nunique() > 1 or per_instance["cpu_model"].nunique() > 1:
        flags.append(f"Koşu {per_instance['session'].nunique()} oturumda, {per_instance['cpu_model'].nunique()} işlemci modelinde yapıldı.")
lines += ["", "## Uyarılar", ""] + (["- " + x for x in flags] if flags else ["Yok."])
lines += ["", "Pilot keşif amaçlıdır: örnek sayısı küçüktür, karşılaştırmalar düzeltilmemiştir ve hiçbiri doğrulayıcı test değildir. "
          "GA-EAX tek çekirdekte koşar; birleştirme kollarında LKH durdurulduktan sonra diğer beş çekirdek boş kalır.", ""]
report = "\n".join(lines)
(OUTPUT_ROOT / "summary" / "report.md").write_text(report, encoding="utf-8")
write_json_atomic(OUTPUT_ROOT / "summary" / "main.json",
                  dict(note=note, suite_complete=bool(suite_complete), instances=len(records), total=len(INSTANCES),
                       primary_arm=PRIMARY_ARM, control_arm=CONTROL_ARM, decision=decision,
                       reference_done=int(sum(r["reference"] is not None for r in records))))
print(report)

In [ ]:
import zipfile


def make_zip(name, include):
    """ZIP önce yerel diskte kurulur, sonra sonuç klasörüne (Drive) kopyalanır."""
    local = WORK_ROOT / name
    count = 0
    with zipfile.ZipFile(local, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            relative = path.relative_to(OUTPUT_ROOT)
            if not path.is_file() or path.suffix in {".zip", ".tmp"} or path.name.endswith(".tmp.npz"):
                continue
            if include(relative):
                z.write(path, str(relative))
                count += 1
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None
    digest = hashlib.sha256(local.read_bytes()).hexdigest()
    shutil.copy2(local, OUTPUT_ROOT / name)
    print(f"{name}: {count} dosya, {local.stat().st_size / 1e6:.1f} MB, SHA-256 {digest}")
    return local


# Sonuç ZIP'i: örnek kayıtları, referans kayıtları, her kolun son turu (uzunlukların bağımsız denetimi için),
# tablolar, rapor ve çalışan kod. Bunu gönderin.
result_zip = make_zip(EXPERIMENT_NAME + "_results.zip",
                      lambda rel: rel.parts[0] in {"instances", "reference", "tours", "summary", "code", "errors"}
                      or rel.name in {"manifest.json", "progress.json"})
if SAVE_TO_DRIVE:
    from google.colab import files
    files.download(str(result_zip))      # tarayıcıya iner
print("ZIP şu klasörde:", OUTPUT_ROOT)